In [ ]:
 from google.colab import drive
 drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
os.makedirs('/content/drive/MyDrive/lockin_project', exist_ok=True)
        # then use the Colab file browser (folder icon, left sidebar) to
        # upload phase0_all.py into that drive/MyDrive/lockin_project folder

In [ ]:
"""
FPGA Lock-In Accelerator - Phase 0 (all steps combined)

This is Steps 1-4 merged into ONE file so nothing breaks if you only upload
a single file to Colab, or if a Colab runtime disconnects and you re-upload
in a hurry. Each original step is still its own section below - nothing was
rewritten, just consolidated.

============================ COLAB SETUP (read this) ========================
Colab's local filesystem is wiped every time the runtime disconnects (idle
timeout, closing the tab, etc.), so re-uploading this file every session
gets old fast. To make it persistent instead:

  1. Run this cell FIRST, once, at the top of your notebook:

        from google.colab import drive
        drive.mount('/content/drive')

     This asks you to authorize Google Drive access (one click), then makes
     your Drive available at /content/drive/MyDrive/.

  2. Then create a folder for this project and save this file there ONCE:

        import os
        os.makedirs('/content/drive/MyDrive/lockin_project', exist_ok=True)
        # then use the Colab file browser (folder icon, left sidebar) to
        # upload phase0_all.py into that drive/MyDrive/lockin_project folder

  3. Every session after that, you only need:

        from google.colab import drive
        drive.mount('/content/drive')
        %cd /content/drive/MyDrive/lockin_project

     and the file is already there - no re-upload needed, and any edits you
     make persist automatically since you're editing the Drive copy directly.

  4. Run it with:  !python phase0_all.py
     Or import individual functions in a notebook cell, e.g.:
        from phase0_all import make_test_signal, ideal_iq_lockin
==============================================================================
"""

import numpy as np


# =============================================================================
# STEPS 1-2: Synthetic test signal + ideal floating-point I/Q lock-in
#
# Pairs with:
#   - SRS App Note #3 "About Lock-In Amplifiers" (pages 1-4 + PSD section)
#   - Rick Lyons "Quadrature Signals: Complex, But Not Complicated"
#
# Goal: prove the core PSD concept works before touching DDS, decimation, or
# fixed-point at all. This is the "golden model" every later stage gets
# checked against.
# =============================================================================

def make_test_signal(fs, f_sig, amplitude, phase_deg, snr_db, duration):
    """
    Generate a noisy sinusoid: the thing a real sensor would hand you.

    fs         : sample rate (Hz)
    f_sig      : signal frequency, i.e. the reference/modulation frequency (Hz)
    amplitude  : true signal amplitude
    phase_deg  : true signal phase (degrees)
    snr_db     : signal-to-noise ratio in dB (can be very negative -
                 that's the whole point of a lock-in)
    duration   : length of the signal (s)
    """
    n = int(fs * duration)
    t = np.arange(n) / fs
    phase_rad = np.deg2rad(phase_deg)

    clean = amplitude * np.cos(2 * np.pi * f_sig * t + phase_rad)

    signal_power = amplitude ** 2 / 2  # average power of a cosine
    snr_linear = 10 ** (snr_db / 10)
    noise_power = signal_power / snr_linear
    noise = np.random.normal(0, np.sqrt(noise_power), n)

    noisy = clean + noise
    return t, noisy


def ideal_iq_lockin(signal, fs, f_ref, lpf_cutoff_hz, lpf_order=4):
    """
    Textbook floating-point I/Q lock-in. No DDS, no fixed-point, no
    decimation tricks - just mix + low-pass, so we know the *algorithm*
    is right before any hardware-shaped constraints get added.

    Returns recovered (amplitude, phase_degrees) as time-series (post-LPF).
    """
    from scipy.signal import butter, sosfiltfilt

    n = len(signal)
    t = np.arange(n) / fs

    ref_i = np.cos(2 * np.pi * f_ref * t)
    ref_q = -np.sin(2 * np.pi * f_ref * t)

    mixed_i = signal * ref_i
    mixed_q = signal * ref_q

    # IMPORTANT: use second-order-sections (sos), not raw (b,a) coefficients.
    # At a very low normalized cutoff the (b,a) polynomial form loses so
    # much precision it produces garbage/overflow. This is the exact
    # real-world reason a hardware lock-in decimates the sample rate down
    # BEFORE the final narrow-band filter, rather than filtering narrow-band
    # at full ADC rate - which is precisely what the CIC+FIR stage is for.
    sos = butter(lpf_order, lpf_cutoff_hz / (fs / 2), btype="low", output="sos")
    i_filt = sosfiltfilt(sos, mixed_i)
    q_filt = sosfiltfilt(sos, mixed_q)

    # Factor of 2 because mixing with a unit-amplitude reference halves
    # the DC term (cos*cos = 0.5*(1+cos(2wt)), the 0.5 survives the LPF).
    magnitude = 2 * np.sqrt(i_filt ** 2 + q_filt ** 2)
    phase = np.degrees(np.arctan2(q_filt, i_filt))

    return magnitude, phase


def steady_state_estimate(series, fraction=0.2):
    """Average over the last `fraction` of samples, where the LPF has settled."""
    tail = series[-int(len(series) * fraction):]
    return np.mean(tail)


def run_steps_1_2():
    np.random.seed(42)  # reproducible - rerun with a different seed to see the variance itself

    fs = 200_000
    f_ref = 1_000
    true_amp = 1.0
    true_phase = 37.0
    duration = 0.5

    print("=== Run 1: fixed LPF cutoff = 20 Hz for every SNR ===")
    print(f"{'SNR (dB)':>10} | {'Recovered Amp':>14} | {'Amp Error %':>11} | "
          f"{'Recovered Phase':>16} | {'Phase Error (deg)':>18}")
    print("-" * 80)

    lpf_cutoff = 20
    for snr_db in [20, 0, -20, -40, -60]:
        t, noisy = make_test_signal(fs, f_ref, true_amp, true_phase, snr_db, duration)
        mag, ph = ideal_iq_lockin(noisy, fs, f_ref, lpf_cutoff)

        amp_est = steady_state_estimate(mag)
        phase_est = steady_state_estimate(ph)

        amp_err_pct = 100 * abs(amp_est - true_amp) / true_amp
        phase_err = abs(((phase_est - true_phase + 180) % 360) - 180)

        print(f"{snr_db:>10} | {amp_est:>14.4f} | {amp_err_pct:>10.2f}% | "
              f"{phase_est:>15.2f}° | {phase_err:>17.3f}°")

    print("\n20 Hz bandwidth is fine down to -20 dB, but at -40/-60 dB the")
    print("output is garbage. That's not a bug - it's the fundamental")
    print("lock-in tradeoff: a fixed low-pass bandwidth only rejects a fixed")
    print("amount of noise power. To dig deeper into the noise floor you")
    print("MUST narrow the bandwidth (average longer). This is exactly why")
    print("adaptive bandwidth is a real feature, not a nice-to-have.")

    print("\n=== Run 2: bandwidth narrowed as SNR drops, duration scaled too ===")
    print(f"{'SNR (dB)':>10} | {'LPF cutoff (Hz)':>16} | {'Duration (s)':>13} | "
          f"{'Recovered Amp':>14} | {'Amp Error %':>11} | {'Phase Error (deg)':>18}")
    print("-" * 105)

    snr_to_cutoff = {20: 20, 0: 20, -20: 10, -40: 1, -60: 0.1}
    for snr_db, cutoff in snr_to_cutoff.items():
        tau = 1 / (2 * np.pi * cutoff)
        run_duration = max(duration, 80 * tau)
        t, noisy = make_test_signal(fs, f_ref, true_amp, true_phase, snr_db, run_duration)
        mag, ph = ideal_iq_lockin(noisy, fs, f_ref, cutoff)

        amp_est = steady_state_estimate(mag)
        phase_est = steady_state_estimate(ph)

        amp_err_pct = 100 * abs(amp_est - true_amp) / true_amp
        phase_err = abs(((phase_est - true_phase + 180) % 360) - 180)

        print(f"{snr_db:>10} | {cutoff:>16} | {run_duration:>13.2f} | {amp_est:>14.4f} | "
              f"{amp_err_pct:>10.2f}% | {phase_err:>17.3f}°")

    print("\nError drops sharply once bandwidth AND duration scale with SNR -")
    print("at -40/-60 dB it goes from >1000% error (Run 1) down to tens of")
    print("percent (Run 2). The remaining error is real statistical variance")
    print("from a single noisy realization, not a bug in the algorithm -")
    print("averaging longer still keeps shrinking it roughly as")
    print("1/sqrt(averaging time), the textbook lock-in noise-rejection law.")
    print("That time/bandwidth/SNR triangle is what adaptive bandwidth automates.")


# =============================================================================
# STEP 3: Direct Digital Synthesis (DDS) phase accumulator
#
# Pairs with: Analog Devices MT-085, "Fundamentals of Direct Digital Synthesis"
#
# Goal: replace the "cheat" np.cos(2*pi*f*t) reference generator with the
# actual structure you'll build in hardware - an integer phase accumulator
# driving a sine lookup, exactly like the Xilinx DDS Compiler IP.
# =============================================================================

class DDSAccumulator:
    """
    Integer phase accumulator, matching hardware structure:
      - phase_bits-wide register, wraps at 2^phase_bits
      - each clock tick, phase_reg += phase_increment (mod 2^phase_bits)
      - output frequency = phase_increment * fclk / 2^phase_bits
      - frequency resolution = fclk / 2^phase_bits
    """

    def __init__(self, fclk, phase_bits=32, lut_bits=10):
        self.fclk = fclk
        self.phase_bits = phase_bits
        self.lut_bits = lut_bits
        self.phase_reg = 0

        lut_size = 2 ** lut_bits
        self.lut_size = lut_size
        self.sine_lut = np.round(
            32767 * np.sin(2 * np.pi * np.arange(lut_size) / lut_size)
        ).astype(np.int32)  # Q1.15-style fixed-point sine table

    def freq_to_increment(self, f_out):
        return round(f_out * (2 ** self.phase_bits) / self.fclk)

    def increment_to_freq(self, phase_increment):
        return phase_increment * self.fclk / (2 ** self.phase_bits)

    def frequency_resolution(self):
        return self.fclk / (2 ** self.phase_bits)

    def generate(self, f_out, n_samples):
        increment = self.freq_to_increment(f_out)
        actual_f_out = self.increment_to_freq(increment)

        mask = (2 ** self.phase_bits) - 1
        samples = np.empty(n_samples, dtype=np.int32)

        phase_reg = self.phase_reg
        shift = self.phase_bits - self.lut_bits
        for i in range(n_samples):
            lut_index = phase_reg >> shift
            samples[i] = self.sine_lut[lut_index]
            phase_reg = (phase_reg + increment) & mask

        self.phase_reg = phase_reg
        return samples, actual_f_out


def run_step_3():
    fclk = 200_000
    dds = DDSAccumulator(fclk, phase_bits=32, lut_bits=10)

    print(f"Phase accumulator width : {dds.phase_bits} bits")
    print(f"LUT size                : {dds.lut_size} entries ({dds.lut_bits}-bit index)")
    print(f"Frequency resolution    : {dds.frequency_resolution():.6f} Hz per LSB")
    print()

    f_requested = 1_000
    n_samples = 2000
    samples, f_actual = dds.generate(f_requested, n_samples)

    print(f"Requested frequency : {f_requested} Hz")
    print(f"Actual frequency    : {f_actual:.6f} Hz "
          f"(error: {abs(f_actual - f_requested):.6f} Hz)")
    print("-> With a 32-bit accumulator the resolution is sub-millihertz.")
    print()

    dds_narrow = DDSAccumulator(fclk, phase_bits=12, lut_bits=10)
    _, f_actual_narrow = dds_narrow.generate(f_requested, n_samples)
    print(f"[12-bit accumulator]  Requested: {f_requested} Hz | "
          f"Actual: {f_actual_narrow:.3f} Hz | "
          f"Resolution: {dds_narrow.frequency_resolution():.3f} Hz/LSB")

    fft = np.fft.rfft(samples * np.hanning(n_samples))
    freqs = np.fft.rfftfreq(n_samples, d=1 / fclk)
    peak_freq = freqs[np.argmax(np.abs(fft))]
    print(f"\nFFT peak of DDS output : {peak_freq:.2f} Hz "
          f"(expected ~{f_actual:.2f} Hz) - confirms the LUT+accumulator "
          f"structure produces a clean tone at the right frequency.")

    print("\n--- Cross-check against Steps 1-2 golden model ---")
    from scipy.signal import butter, sosfiltfilt

    true_amp, true_phase, snr_db, duration = 1.0, 37.0, 0, 0.5
    t, noisy = make_test_signal(fclk, f_requested, true_amp, true_phase, snr_db, duration)

    n = len(noisy)
    dds_check = DDSAccumulator(fclk, phase_bits=32, lut_bits=10)
    dds_check.phase_reg = (2 ** 32) // 4  # +90 deg -> sin(theta+90) = cos(theta)
    ref_samples, _ = dds_check.generate(f_requested, n)
    ref_i = ref_samples / 32767.0

    dds_check_q = DDSAccumulator(fclk, phase_bits=32, lut_bits=10)
    dds_check_q.phase_reg = (2 ** 32) // 2  # +180 deg -> sin(theta+180) = -sin(theta)
    ref_q_samples, _ = dds_check_q.generate(f_requested, n)
    ref_q = ref_q_samples / 32767.0

    mixed_i = noisy * ref_i
    mixed_q = noisy * ref_q
    sos = butter(4, 20 / (fclk / 2), btype="low", output="sos")
    i_filt = sosfiltfilt(sos, mixed_i)
    q_filt = sosfiltfilt(sos, mixed_q)
    magnitude = 2 * np.sqrt(i_filt ** 2 + q_filt ** 2)
    phase = np.degrees(np.arctan2(q_filt, i_filt))

    print(f"DDS-driven recovery -> Amplitude: {steady_state_estimate(magnitude):.4f} "
          f"(true {true_amp}) | Phase: {steady_state_estimate(phase):.2f} deg "
          f"(true {true_phase})")
    print("If this is close to the Steps 1-2 numbers at SNR=0 dB, the DDS")
    print("reference is a drop-in replacement for the floating-point one.")


# =============================================================================
# STEP 4: CIC (Cascaded Integrator-Comb) + compensating FIR decimation
#
# Pairs with: Rick Lyons, "Understanding Cascaded Integrator-Comb Filters"
#
# Goal: replace the "cheat" butter/sosfiltfilt low-pass filter with the
# actual filter structure you'll build in hardware - a CIC decimator (no
# multipliers, just integrators/combs) followed by a short compensating FIR.
# =============================================================================

def cic_decimate(x, decimation, n_stages=4, diff_delay=1):
    """
    CIC decimator: N cascaded integrators, downsample by R, then N cascaded
    comb (differentiator) stages. No multipliers anywhere.
    """
    y = x.astype(np.float64).copy()

    for _ in range(n_stages):
        y = np.cumsum(y)

    y = y[::decimation]

    for _ in range(n_stages):
        y = y - np.concatenate([np.zeros(diff_delay), y[:-diff_delay]])

    dc_gain = decimation ** n_stages
    return y / dc_gain


def cic_frequency_response(decimation, n_stages, diff_delay, n_freqs=2000):
    """Magnitude response of a CIC filter (normalized frequency 0 to 0.5 of
    the INPUT sample rate), used to see the passband droop this causes."""
    f = np.linspace(1e-6, 0.5, n_freqs)
    num = np.sin(np.pi * f * decimation * diff_delay)
    den = np.sin(np.pi * f * diff_delay)
    den = np.where(np.abs(den) < 1e-12, 1e-12, den)
    h = np.abs(num / den) ** n_stages
    h = h / h[0]
    return f, h


def design_compensating_fir(decimation, n_stages, diff_delay, numtaps, fs_out):
    """
    Short FIR designed to invert the CIC's passband droop across the full
    output Nyquist band.

    firls's `bands` argument must be pairs of (start, end) frequencies with
    NO gaps between consecutive bands if you want one continuous curve - a
    flat list of N points is read as N/2 independent (band, transition)
    pairs, which silently drops half the intended shape. Each interior
    breakpoint must appear twice (end of one band, start of the next) to
    get a genuinely continuous piecewise-linear desired response.
    """
    from scipy.signal import firls

    n_points = 60
    out_nyquist_norm = 0.5 / decimation
    f_input_frac = np.linspace(1e-6, out_nyquist_norm, n_points)

    _, h_full = cic_frequency_response(decimation, n_stages, diff_delay, n_freqs=4000)
    f_full = np.linspace(1e-6, 0.5, 4000)
    h_cic_at_points = np.interp(f_input_frac, f_full, h_full)

    desired_at_points = 1.0 / h_cic_at_points
    desired_at_points = np.clip(desired_at_points, None, 3 * desired_at_points[0])

    freq_norm = f_input_frac / out_nyquist_norm

    bands = np.repeat(freq_norm, 2)[1:-1]
    desired = np.repeat(desired_at_points, 2)[1:-1]

    taps = firls(numtaps, bands, desired, fs=2.0)

    # firls doesn't force unity DC gain - fix it up explicitly
    taps = taps / np.sum(taps) * desired_at_points[0]
    return taps


def run_step_4():
    from scipy.signal import lfilter

    fs = 200_000
    f_ref = 1_000
    true_amp, true_phase = 1.0, 37.0
    snr_db = 0
    duration = 0.1

    t, noisy = make_test_signal(fs, f_ref, true_amp, true_phase, snr_db, duration)

    ref_i = np.cos(2 * np.pi * f_ref * t)
    ref_q = -np.sin(2 * np.pi * f_ref * t)
    mixed_i = noisy * ref_i
    mixed_q = noisy * ref_q

    decimation = 100
    n_stages = 4
    diff_delay = 1

    print(f"Input rate  : {fs} Hz")
    print(f"Decimation  : {decimation}x -> output rate {fs/decimation:.0f} Hz")
    print(f"CIC stages  : {n_stages}, comb delay {diff_delay}")
    print()

    i_cic = cic_decimate(mixed_i, decimation, n_stages, diff_delay)
    q_cic = cic_decimate(mixed_q, decimation, n_stages, diff_delay)

    magnitude_raw = 2 * np.sqrt(i_cic ** 2 + q_cic ** 2)
    phase_raw = np.degrees(np.arctan2(q_cic, i_cic))

    tail = int(len(magnitude_raw) * 0.3)
    print("=== CIC only (no compensation) ===")
    print(f"Recovered amplitude : {np.mean(magnitude_raw[-tail:]):.4f} (true {true_amp})")
    print(f"Recovered phase     : {np.mean(phase_raw[-tail:]):.2f} deg (true {true_phase})")

    f_resp, h_resp = cic_frequency_response(decimation, n_stages, diff_delay)
    out_nyquist = 0.5 / decimation
    passband_mask = f_resp <= out_nyquist
    droop_db = 20 * np.log10(h_resp[passband_mask][-1] / h_resp[passband_mask][0])
    print(f"\nCIC passband droop (DC to output Nyquist): {droop_db:.2f} dB")

    print("\n=== Adding a short compensating FIR ===")
    fir_taps = design_compensating_fir(decimation, n_stages, diff_delay,
                                        numtaps=21, fs_out=fs / decimation)
    i_comp = lfilter(fir_taps, 1.0, i_cic)
    q_comp = lfilter(fir_taps, 1.0, q_cic)

    magnitude_comp = 2 * np.sqrt(i_comp ** 2 + q_comp ** 2)
    phase_comp = np.degrees(np.arctan2(q_comp, i_comp))

    print(f"Recovered amplitude : {np.mean(magnitude_comp[-tail:]):.4f} (true {true_amp})")
    print(f"Recovered phase     : {np.mean(phase_comp[-tail:]):.2f} deg (true {true_phase})")
    print(f"FIR taps used       : {len(fir_taps)} (cheap on FPGA resources)")
    print()
    print("Honest note: CIC-only already recovered amplitude well here because")
    print("this test signal has a CONSTANT amplitude/phase after mixing. CIC")
    print("droop only affects frequency content spread across the passband;")
    print("the compensating FIR earns its keep when the demodulated envelope")
    print("varies quickly in time. Droop at output Nyquist is decimation-")
    print("independent - a fixed property of CIC order/delay. Worth validating")
    print("with a modulated-amplitude input in Phase 3, not just a steady tone.")


# =============================================================================
# Run everything
# =============================================================================

if __name__ == "__main__":
    print("############################################")
    print("# Phase 0, Steps 1-2: Signal + Ideal Lock-In #")
    print("############################################\n")
    run_steps_1_2()

    print("\n\n############################################")
    print("# Phase 0, Step 3: DDS Phase Accumulator     #")
    print("############################################\n")
    run_step_3()

    print("\n\n############################################")
    print("# Phase 0, Step 4: CIC + Compensating FIR    #")
    print("############################################\n")
    run_step_4()

############################################
# Phase 0, Steps 1-2: Signal + Ideal Lock-In #
############################################

=== Run 1: fixed LPF cutoff = 20 Hz for every SNR ===
  SNR (dB) |  Recovered Amp | Amp Error % |  Recovered Phase |  Phase Error (deg)
--------------------------------------------------------------------------------
        20 |         0.9997 |       0.03% |           36.91° |             0.086°
         0 |         0.9874 |       1.26% |           36.60° |             0.402°
       -20 |         1.1169 |      11.69% |           39.91° |             2.911°
       -40 |         1.7364 |      73.64% |          -37.46° |            74.459°
       -60 |        19.7345 |    1873.45% |           23.50° |            13.501°

20 Hz bandwidth is fine down to -20 dB, but at -40/-60 dB the
output is garbage. That's not a bug - it's the fundamental
lock-in tradeoff: a fixed low-pass bandwidth only rejects a fixed
amount of noise power. To dig deeper into the n

In [ ]:
import numpy as np

def build_atan_table(n_iterations):
    """Precompute atan(2^-i) for i = 0..n_iterations-1, in degrees.
    In real hardware this is a small ROM, not computed at runtime."""
    return [np.degrees(np.arctan(2.0 ** -i)) for i in range(n_iterations)]


def cordic_gain(n_iterations):
    """CORDIC gain factor - each micro-rotation scales vector length by
    sqrt(1 + 2^-2i); the cumulative product converges to ~1.6467.
    We divide it back out to get true magnitude."""
    gain = 1.0
    for i in range(n_iterations):
        gain *= np.sqrt(1 + 2.0 ** (-2 * i))
    return gain


def cordic_vectoring(x, y, n_iterations, atan_table=None):
    """
    Single-sample CORDIC vectoring mode: rotate (x, y) toward the x-axis.
    Returns (magnitude, angle_degrees).

    Native CORDIC vectoring only converges for angles within -90..+90
    degrees of the positive x-axis. To cover all four quadrants, we do an
    explicit pre-rotation: if x is negative, rotate by 180 degrees first
    and add/subtract 180 at the end. Standard "quadrant correction" step
    in every real CORDIC implementation, hardware or software.
    """
    if atan_table is None:
        atan_table = build_atan_table(n_iterations)

    angle_offset = 0.0
    if x < 0:
        x, y = -x, -y
        angle_offset = 180.0 if y >= 0 else -180.0

    cur_x, cur_y = float(x), float(y)
    angle_acc = 0.0

    for i in range(n_iterations):
        step_angle = atan_table[i]
        if cur_y < 0:
            new_x = cur_x - cur_y * (2.0 ** -i)
            new_y = cur_y + cur_x * (2.0 ** -i)
            angle_acc -= step_angle
        else:
            new_x = cur_x + cur_y * (2.0 ** -i)
            new_y = cur_y - cur_x * (2.0 ** -i)
            angle_acc += step_angle
        cur_x, cur_y = new_x, new_y

    magnitude = cur_x / cordic_gain(n_iterations)
    angle = angle_acc + angle_offset
    # normalize to (-180, 180] - quadrant correction can otherwise produce
    # mathematically-equivalent but confusing results like -225 instead of 135
    angle = ((angle + 180) % 360) - 180

    return magnitude, angle


def cordic_vectoring_array(x_arr, y_arr, n_iterations):
    """Vectorized convenience wrapper - runs cordic_vectoring per-sample."""
    atan_table = build_atan_table(n_iterations)
    mags = np.empty(len(x_arr))
    angles = np.empty(len(x_arr))
    for i in range(len(x_arr)):
        mags[i], angles[i] = cordic_vectoring(x_arr[i], y_arr[i], n_iterations, atan_table)
    return mags, angles


# === Sanity check: known (x, y) pairs across all four quadrants ===
print("=== Sanity check: known (x, y) pairs across all four quadrants ===")
test_cases = [
    (1.0, 0.0, 1.0, 0.0),
    (0.0, 1.0, 1.0, 90.0),
    (-1.0, 0.0, 1.0, 180.0),
    (0.0, -1.0, 1.0, -90.0),
    (1.0, 1.0, np.sqrt(2), 45.0),
    (-1.0, 1.0, np.sqrt(2), 135.0),
    (-1.0, -1.0, np.sqrt(2), -135.0),
    (1.0, -1.0, np.sqrt(2), -45.0),
    (3.0, 4.0, 5.0, np.degrees(np.arctan2(4, 3))),
]

n_iter = 16
print(f"Using {n_iter} CORDIC iterations\n")
print(f"{'x':>6} {'y':>6} | {'Expected Mag':>13} {'Got Mag':>10} | "
      f"{'Expected Ang':>13} {'Got Ang':>10}")
print("-" * 70)
for x, y, exp_mag, exp_ang in test_cases:
    mag, ang = cordic_vectoring(x, y, n_iter)
    print(f"{x:>6.1f} {y:>6.1f} | {exp_mag:>13.4f} {mag:>10.4f} | "
          f"{exp_ang:>13.2f} {ang:>10.2f}")

# === Error vs iteration count ===
print("\n=== Error vs iteration count ===")
x_test, y_test = 1.0, 2.0
true_mag = np.hypot(x_test, y_test)
true_ang = np.degrees(np.arctan2(y_test, x_test))
print(f"\n{'Iterations':>10} | {'Mag Error':>12} | {'Angle Error (deg)':>18}")
print("-" * 50)
for n_it in [2, 4, 6, 8, 10, 12, 16, 20, 24]:
    mag, ang = cordic_vectoring(x_test, y_test, n_it)
    print(f"{n_it:>10} | {abs(mag - true_mag):>12.6f} | {abs(ang - true_ang):>18.6f}")

# === Cross-check against Phase 0's I/Q lock-in output ===
# (uses make_test_signal / ideal_iq_lockin / steady_state_estimate from Cell 1 -
#  make sure Cell 1 has already been run in this session)
print("\n=== Cross-check against Phase 0's I/Q lock-in output ===")
from scipy.signal import butter, sosfiltfilt

np.random.seed(42)
fs, f_ref = 200_000, 1_000
true_amp, true_phase, snr_db, duration = 1.0, 37.0, 0, 0.5
lpf_cutoff = 20

t, noisy = make_test_signal(fs, f_ref, true_amp, true_phase, snr_db, duration)
ref_i = np.cos(2 * np.pi * f_ref * t)
ref_q = -np.sin(2 * np.pi * f_ref * t)
mixed_i = noisy * ref_i
mixed_q = noisy * ref_q
sos = butter(4, lpf_cutoff / (fs / 2), btype="low", output="sos")
i_filt = sosfiltfilt(sos, mixed_i)
q_filt = sosfiltfilt(sos, mixed_q)

mag_cordic, ang_cordic = cordic_vectoring_array(i_filt, q_filt, n_iterations=16)
mag_cordic *= 2  # same mixing-gain correction as Phase 0

tail = int(len(mag_cordic) * 0.2)
print(f"CORDIC-based recovery -> Amplitude: {np.mean(mag_cordic[-tail:]):.4f} "
      f"(true {true_amp}) | Phase: {np.mean(ang_cordic[-tail:]):.2f} deg "
      f"(true {true_phase})")

mag_ref, ang_ref = ideal_iq_lockin(noisy, fs, f_ref, lpf_cutoff)
print(f"Reference (np.arctan2/hypot) -> Amplitude: {steady_state_estimate(mag_ref):.4f} "
      f"| Phase: {steady_state_estimate(ang_ref):.2f} deg")

=== Sanity check: known (x, y) pairs across all four quadrants ===
Using 16 CORDIC iterations

     x      y |  Expected Mag    Got Mag |  Expected Ang    Got Ang
----------------------------------------------------------------------
   1.0    0.0 |        1.0000     1.0000 |          0.00      -0.00
   0.0    1.0 |        1.0000     1.0000 |         90.00      90.00
  -1.0    0.0 |        1.0000     1.0000 |        180.00     180.00
   0.0   -1.0 |        1.0000     1.0000 |        -90.00     -90.00
   1.0    1.0 |        1.4142     1.4142 |         45.00      45.00
  -1.0    1.0 |        1.4142     1.4142 |        135.00     135.00
  -1.0   -1.0 |        1.4142     1.4142 |       -135.00    -135.00
   1.0   -1.0 |        1.4142     1.4142 |        -45.00     -45.00
   3.0    4.0 |        5.0000     5.0000 |         53.13      53.13

=== Error vs iteration count ===

Iterations |    Mag Error |  Angle Error (deg)
--------------------------------------------------
         2 |     0.02

In [ ]:
import numpy as np

def build_atan_lut(table_bits):
    """Store atan(r) for r in [0,1], where r = min(|x|,|y|)/max(|x|,|y|).
    This ratio ALWAYS falls in [0,1] regardless of the actual angle, which
    is why one small table can cover the whole circle - the octant
    reduction below handles getting back to the true angle."""
    size = 2 ** table_bits
    r_grid = np.linspace(0, 1, size)
    atan_vals = np.degrees(np.arctan(r_grid))
    return r_grid, atan_vals


def build_mag_scale_lut(table_bits):
    """Store sqrt(1+r^2) for the same r grid - this scales max(|x|,|y|)
    up to the true magnitude."""
    size = 2 ** table_bits
    r_grid = np.linspace(0, 1, size)
    scale_vals = np.sqrt(1 + r_grid ** 2)
    return r_grid, scale_vals


def lut_lookup_interp(r, r_grid, table):
    """Linear interpolation between the two nearest table entries -
    this is what lets a small table still give a smooth, accurate answer
    for r values that fall BETWEEN grid points."""
    idx_f = r * (len(r_grid) - 1)
    idx0 = int(np.floor(idx_f))
    idx1 = min(idx0 + 1, len(r_grid) - 1)
    frac = idx_f - idx0
    return table[idx0] * (1 - frac) + table[idx1] * frac


def lut_vectoring(x, y, r_grid, atan_table, r_grid_mag, mag_table):
    """
    LUT-based magnitude/phase extraction.

    Same octant-reduction idea as CORDIC's quadrant correction, but instead
    of iteratively rotating toward the answer, we reduce (x,y) down to a
    single ratio r = min(|x|,|y|)/max(|x|,|y|) in [0,1], look up atan(r)
    directly from a precomputed table, then undo the reduction to recover
    the true angle and magnitude.
    """
    ax, ay = abs(x), abs(y)
    if ax == 0 and ay == 0:
        return 0.0, 0.0

    if ax >= ay:
        r = ay / ax if ax != 0 else 0.0
        base_angle_from_x_axis = True
    else:
        r = ax / ay
        base_angle_from_x_axis = False

    looked_up_angle = lut_lookup_interp(r, r_grid, atan_table)
    scale = lut_lookup_interp(r, r_grid_mag, mag_table)
    magnitude = max(ax, ay) * scale

    if base_angle_from_x_axis:
        angle = looked_up_angle
    else:
        angle = 90 - looked_up_angle

    if x >= 0 and y >= 0:
        pass
    elif x < 0 and y >= 0:
        angle = 180 - angle
    elif x < 0 and y < 0:
        angle = -(180 - angle)
    else:  # x >= 0, y < 0
        angle = -angle

    return magnitude, angle


# === Sanity check: same 9 known quadrant cases as CORDIC ===
print("=== Sanity check: known (x, y) pairs across all four quadrants ===")
r_grid, atan_table = build_atan_lut(8)
r_grid_mag, mag_table = build_mag_scale_lut(8)

test_cases = [
    (1.0, 0.0, 1.0, 0.0),
    (0.0, 1.0, 1.0, 90.0),
    (-1.0, 0.0, 1.0, 180.0),
    (0.0, -1.0, 1.0, -90.0),
    (1.0, 1.0, np.sqrt(2), 45.0),
    (-1.0, 1.0, np.sqrt(2), 135.0),
    (-1.0, -1.0, np.sqrt(2), -135.0),
    (1.0, -1.0, np.sqrt(2), -45.0),
    (3.0, 4.0, 5.0, np.degrees(np.arctan2(4, 3))),
]
print(f"{'x':>6} {'y':>6} | {'Expected Mag':>13} {'Got Mag':>10} | "
      f"{'Expected Ang':>13} {'Got Ang':>10}")
print("-" * 70)
for x, y, exp_mag, exp_ang in test_cases:
    mag, ang = lut_vectoring(x, y, r_grid, atan_table, r_grid_mag, mag_table)
    print(f"{x:>6.1f} {y:>6.1f} | {exp_mag:>13.4f} {mag:>10.4f} | "
          f"{exp_ang:>13.2f} {ang:>10.2f}")

# === Table size vs accuracy vs memory - the real design tradeoff ===
print("\n=== Table size vs accuracy vs memory ===")
np.random.seed(0)
n_test = 2000
test_angles = np.random.uniform(-180, 180, n_test)
test_r = np.random.uniform(0.1, 10, n_test)
xs = test_r * np.cos(np.radians(test_angles))
ys = test_r * np.sin(np.radians(test_angles))

print(f"{'Table bits':>10} | {'Table entries':>13} | {'Memory (bytes, 16-bit)':>22} | "
      f"{'Max angle err (deg)':>19} | {'Max mag err %':>13}")
print("-" * 90)
for bits in [2, 4, 6, 8, 10, 12]:
    r_grid, atan_table = build_atan_lut(bits)
    r_grid_mag, mag_table = build_mag_scale_lut(bits)
    max_ang_err, max_mag_err = 0, 0
    for x, y, true_r, true_ang in zip(xs, ys, test_r, test_angles):
        mag, ang = lut_vectoring(x, y, r_grid, atan_table, r_grid_mag, mag_table)
        ang_err = abs(((ang - true_ang + 180) % 360) - 180)
        mag_err = abs(mag - true_r) / true_r * 100
        max_ang_err = max(max_ang_err, ang_err)
        max_mag_err = max(max_mag_err, mag_err)
    entries = 2 ** bits
    mem_bytes = entries * 2 * 2  # 2 tables (atan + magnitude scale), 16-bit each
    print(f"{bits:>10} | {entries:>13} | {mem_bytes:>22} | "
          f"{max_ang_err:>19.4f} | {max_mag_err:>13.4f}")

# === Cross-check against Phase 0's I/Q lock-in output (needs Cell 1 run first) ===
print("\n=== Cross-check against Phase 0's I/Q lock-in output ===")
from scipy.signal import butter, sosfiltfilt

np.random.seed(42)
fs, f_ref = 200_000, 1_000
true_amp, true_phase, snr_db, duration = 1.0, 37.0, 0, 0.5
lpf_cutoff = 20

t, noisy = make_test_signal(fs, f_ref, true_amp, true_phase, snr_db, duration)
ref_i = np.cos(2 * np.pi * f_ref * t)
ref_q = -np.sin(2 * np.pi * f_ref * t)
mixed_i = noisy * ref_i
mixed_q = noisy * ref_q
sos = butter(4, lpf_cutoff / (fs / 2), btype="low", output="sos")
i_filt = sosfiltfilt(sos, mixed_i)
q_filt = sosfiltfilt(sos, mixed_q)

r_grid, atan_table = build_atan_lut(8)
r_grid_mag, mag_table = build_mag_scale_lut(8)
mags, angs = [], []
for xi, yi in zip(i_filt, q_filt):
    m, a = lut_vectoring(xi, yi, r_grid, atan_table, r_grid_mag, mag_table)
    mags.append(m)
    angs.append(a)
mags = np.array(mags) * 2
angs = np.array(angs)

tail = int(len(mags) * 0.2)
print(f"LUT-based recovery -> Amplitude: {np.mean(mags[-tail:]):.4f} "
      f"(true {true_amp}) | Phase: {np.mean(angs[-tail:]):.2f} deg (true {true_phase})")

mag_ref, ang_ref = ideal_iq_lockin(noisy, fs, f_ref, lpf_cutoff)
print(f"Reference (np.arctan2/hypot) -> Amplitude: {steady_state_estimate(mag_ref):.4f} "
      f"| Phase: {steady_state_estimate(ang_ref):.2f} deg")

=== Sanity check: known (x, y) pairs across all four quadrants ===
     x      y |  Expected Mag    Got Mag |  Expected Ang    Got Ang
----------------------------------------------------------------------
   1.0    0.0 |        1.0000     1.0000 |          0.00       0.00
   0.0    1.0 |        1.0000     1.0000 |         90.00      90.00
  -1.0    0.0 |        1.0000     1.0000 |        180.00     180.00
   0.0   -1.0 |        1.0000     1.0000 |        -90.00     -90.00
   1.0    1.0 |        1.4142     1.4142 |         45.00      45.00
  -1.0    1.0 |        1.4142     1.4142 |        135.00     135.00
  -1.0   -1.0 |        1.4142     1.4142 |       -135.00    -135.00
   1.0   -1.0 |        1.4142     1.4142 |        -45.00     -45.00
   3.0    4.0 |        5.0000     5.0000 |         53.13      53.13

=== Table size vs accuracy vs memory ===
Table bits | Table entries | Memory (bytes, 16-bit) | Max angle err (deg) | Max mag err %
--------------------------------------------------

In [ ]:
import numpy as np

def fit_atan_poly(order):
    """Fit a Chebyshev polynomial to atan(r) over r in [0,1].
    Chebyshev fitting (vs plain polyfit) spreads error evenly across the
    range instead of concentrating it at the edges - the textbook reason
    minimax-style approximations use Chebyshev basis."""
    r_fit = np.linspace(0, 1, 500)
    atan_vals = np.degrees(np.arctan(r_fit))
    coeffs = np.polynomial.chebyshev.chebfit(r_fit, atan_vals, order)
    return coeffs


def fit_mag_scale_poly(order):
    """Fit a Chebyshev polynomial to sqrt(1+r^2) over r in [0,1]."""
    r_fit = np.linspace(0, 1, 500)
    scale_vals = np.sqrt(1 + r_fit ** 2)
    coeffs = np.polynomial.chebyshev.chebfit(r_fit, scale_vals, order)
    return coeffs


def poly_eval(r, coeffs):
    return np.polynomial.chebyshev.chebval(r, coeffs)


def poly_vectoring(x, y, atan_coeffs, mag_coeffs):
    """
    Polynomial-approximation magnitude/phase extraction.

    Same octant-reduction structure as CORDIC and LUT - only the
    r -> angle and r -> scale steps differ (here: evaluating a short
    polynomial via Horner's method, which needs `order` multiplies and
    `order` adds, instead of CORDIC's shift-add rotations or LUT's
    memory lookup).
    """
    ax, ay = abs(x), abs(y)
    if ax == 0 and ay == 0:
        return 0.0, 0.0

    if ax >= ay:
        r = ay / ax if ax != 0 else 0.0
        base_angle_from_x_axis = True
    else:
        r = ax / ay
        base_angle_from_x_axis = False

    looked_up_angle = poly_eval(r, atan_coeffs)
    scale = poly_eval(r, mag_coeffs)
    magnitude = max(ax, ay) * scale

    if base_angle_from_x_axis:
        angle = looked_up_angle
    else:
        angle = 90 - looked_up_angle

    if x >= 0 and y >= 0:
        pass
    elif x < 0 and y >= 0:
        angle = 180 - angle
    elif x < 0 and y < 0:
        angle = -(180 - angle)
    else:  # x >= 0, y < 0
        angle = -angle

    return magnitude, angle


# === Sanity check: same 9 known quadrant cases as CORDIC and LUT ===
print("=== Sanity check: known (x, y) pairs across all four quadrants ===")
atan_coeffs = fit_atan_poly(5)
mag_coeffs = fit_mag_scale_poly(5)

test_cases = [
    (1.0, 0.0, 1.0, 0.0),
    (0.0, 1.0, 1.0, 90.0),
    (-1.0, 0.0, 1.0, 180.0),
    (0.0, -1.0, 1.0, -90.0),
    (1.0, 1.0, np.sqrt(2), 45.0),
    (-1.0, 1.0, np.sqrt(2), 135.0),
    (-1.0, -1.0, np.sqrt(2), -135.0),
    (1.0, -1.0, np.sqrt(2), -45.0),
    (3.0, 4.0, 5.0, np.degrees(np.arctan2(4, 3))),
]
print(f"{'x':>6} {'y':>6} | {'Expected Mag':>13} {'Got Mag':>10} | "
      f"{'Expected Ang':>13} {'Got Ang':>10}")
print("-" * 70)
for x, y, exp_mag, exp_ang in test_cases:
    mag, ang = poly_vectoring(x, y, atan_coeffs, mag_coeffs)
    print(f"{x:>6.1f} {y:>6.1f} | {exp_mag:>13.4f} {mag:>10.4f} | "
          f"{exp_ang:>13.2f} {ang:>10.2f}")
print("\n(Tiny magnitude error here, e.g. 5.0000 vs 4.9998, is the")
print("polynomial's own approximation error at this order - expected,")
print("not a bug. It shrinks as order increases, shown below.)")

# === Polynomial order vs accuracy vs multiplier cost - the real tradeoff ===
print("\n=== Polynomial order vs accuracy vs multiplier cost ===")
np.random.seed(0)
n_test = 2000
test_angles = np.random.uniform(-180, 180, n_test)
test_r = np.random.uniform(0.1, 10, n_test)
xs = test_r * np.cos(np.radians(test_angles))
ys = test_r * np.sin(np.radians(test_angles))

print(f"{'Order':>6} | {'Multiplies (Horner)':>19} | {'Max angle err (deg)':>19} | {'Max mag err %':>13}")
print("-" * 70)
for order in [1, 2, 3, 4, 5, 6, 8]:
    atan_c = fit_atan_poly(order)
    mag_c = fit_mag_scale_poly(order)
    max_ang_err, max_mag_err = 0, 0
    for x, y, true_r, true_ang in zip(xs, ys, test_r, test_angles):
        mag, ang = poly_vectoring(x, y, atan_c, mag_c)
        ang_err = abs(((ang - true_ang + 180) % 360) - 180)
        mag_err = abs(mag - true_r) / true_r * 100
        max_ang_err = max(max_ang_err, ang_err)
        max_mag_err = max(max_mag_err, mag_err)
    # Horner's method needs `order` multiplies + `order` adds per polynomial,
    # x2 since we evaluate both the atan poly and the magnitude-scale poly
    multiplies = 2 * order
    print(f"{order:>6} | {multiplies:>19} | {max_ang_err:>19.4f} | {max_mag_err:>13.4f}")

# === Cross-check against Phase 0's I/Q lock-in output (needs Cell 1 run first) ===
print("\n=== Cross-check against Phase 0's I/Q lock-in output ===")
from scipy.signal import butter, sosfiltfilt

np.random.seed(42)
fs, f_ref = 200_000, 1_000
true_amp, true_phase, snr_db, duration = 1.0, 37.0, 0, 0.5
lpf_cutoff = 20

t, noisy = make_test_signal(fs, f_ref, true_amp, true_phase, snr_db, duration)
ref_i = np.cos(2 * np.pi * f_ref * t)
ref_q = -np.sin(2 * np.pi * f_ref * t)
mixed_i = noisy * ref_i
mixed_q = noisy * ref_q
sos = butter(4, lpf_cutoff / (fs / 2), btype="low", output="sos")
i_filt = sosfiltfilt(sos, mixed_i)
q_filt = sosfiltfilt(sos, mixed_q)

atan_c = fit_atan_poly(5)
mag_c = fit_mag_scale_poly(5)
mags, angs = [], []
for xi, yi in zip(i_filt, q_filt):
    m, a = poly_vectoring(xi, yi, atan_c, mag_c)
    mags.append(m)
    angs.append(a)
mags = np.array(mags) * 2
angs = np.array(angs)

tail = int(len(mags) * 0.2)
print(f"Poly-based recovery -> Amplitude: {np.mean(mags[-tail:]):.4f} "
      f"(true {true_amp}) | Phase: {np.mean(angs[-tail:]):.2f} deg (true {true_phase})")

mag_ref, ang_ref = ideal_iq_lockin(noisy, fs, f_ref, lpf_cutoff)
print(f"Reference (np.arctan2/hypot) -> Amplitude: {steady_state_estimate(mag_ref):.4f} "
      f"| Phase: {steady_state_estimate(ang_ref):.2f} deg")

=== Sanity check: known (x, y) pairs across all four quadrants ===
     x      y |  Expected Mag    Got Mag |  Expected Ang    Got Ang
----------------------------------------------------------------------
   1.0    0.0 |        1.0000     1.0000 |          0.00       0.00
   0.0    1.0 |        1.0000     1.0000 |         90.00      90.00
  -1.0    0.0 |        1.0000     1.0000 |        180.00     180.00
   0.0   -1.0 |        1.0000     1.0000 |        -90.00     -90.00
   1.0    1.0 |        1.4142     1.4142 |         45.00      45.00
  -1.0    1.0 |        1.4142     1.4142 |        135.00     135.00
  -1.0   -1.0 |        1.4142     1.4142 |       -135.00    -135.00
   1.0   -1.0 |        1.4142     1.4142 |        -45.00     -45.00
   3.0    4.0 |        5.0000     5.0000 |         53.13      53.13

(Tiny magnitude error here, e.g. 5.0000 vs 4.9998, is the
polynomial's own approximation error at this order - expected,
not a bug. It shrinks as order increases, shown below.)

===

In [ ]:
import numpy as np

np.random.seed(0)
n_test = 5000
test_angles = np.random.uniform(-180, 180, n_test)
test_r = np.random.uniform(0.1, 10, n_test)
xs = test_r * np.cos(np.radians(test_angles))
ys = test_r * np.sin(np.radians(test_angles))

# (method name, parameter name, parameter value) - same test vectors for all
configs = [
    ("CORDIC",     "iterations", 4),
    ("CORDIC",     "iterations", 8),
    ("CORDIC",     "iterations", 12),
    ("CORDIC",     "iterations", 16),
    ("LUT",        "table_bits", 4),
    ("LUT",        "table_bits", 6),
    ("LUT",        "table_bits", 8),
    ("LUT",        "table_bits", 10),
    ("Polynomial", "order",      2),
    ("Polynomial", "order",      4),
    ("Polynomial", "order",      6),
    ("Polynomial", "order",      8),
]

print(f"{'Method':>10} | {'Param':>12} | {'Est. HW cost':>28} | "
      f"{'Max angle err (deg)':>19} | {'Max mag err %':>13}")
print("-" * 100)

for method, param_name, param_val in configs:
    max_ang_err, max_mag_err = 0, 0

    if method == "CORDIC":
        atan_table = build_atan_table(param_val)
        for x, y, true_r, true_ang in zip(xs, ys, test_r, test_angles):
            mag, ang = cordic_vectoring(x, y, param_val, atan_table)
            ang_err = abs(((ang - true_ang + 180) % 360) - 180)
            mag_err = abs(mag - true_r) / true_r * 100
            max_ang_err = max(max_ang_err, ang_err)
            max_mag_err = max(max_mag_err, mag_err)
        cost_str = f"{param_val} pipeline stages"

    elif method == "LUT":
        r_grid, atan_tab = build_atan_lut(param_val)
        r_grid_m, mag_tab = build_mag_scale_lut(param_val)
        for x, y, true_r, true_ang in zip(xs, ys, test_r, test_angles):
            mag, ang = lut_vectoring(x, y, r_grid, atan_tab, r_grid_m, mag_tab)
            ang_err = abs(((ang - true_ang + 180) % 360) - 180)
            mag_err = abs(mag - true_r) / true_r * 100
            max_ang_err = max(max_ang_err, ang_err)
            max_mag_err = max(max_mag_err, mag_err)
        entries = 2 ** param_val
        cost_str = f"{entries * 2} entries x2 tables"

    else:  # Polynomial
        atan_c = fit_atan_poly(param_val)
        mag_c = fit_mag_scale_poly(param_val)
        for x, y, true_r, true_ang in zip(xs, ys, test_r, test_angles):
            mag, ang = poly_vectoring(x, y, atan_c, mag_c)
            ang_err = abs(((ang - true_ang + 180) % 360) - 180)
            mag_err = abs(mag - true_r) / true_r * 100
            max_ang_err = max(max_ang_err, ang_err)
            max_mag_err = max(max_mag_err, mag_err)
        cost_str = f"{2 * param_val} multiplies"

    print(f"{method:>10} | {param_val:>12} | {cost_str:>28} | "
          f"{max_ang_err:>19.4f} | {max_mag_err:>13.4f}")

print("\nRead this as: for a similar accuracy target, what does each method")
print("actually cost? e.g. ~0.02 deg error needs CORDIC@12 (12 pipeline")
print("stages, pure logic), LUT@6 (128 entries, pure memory), or")
print("Polynomial@6 (12 multiplies, pure DSP slices) - three genuinely")
print("different hardware resource types for roughly the same accuracy.")
print("This table is your Phase 1 paper figure.")

    Method |        Param |                 Est. HW cost | Max angle err (deg) | Max mag err %
----------------------------------------------------------------------------------------------------
    CORDIC |            4 |            4 pipeline stages |              7.1244 |        0.7721
    CORDIC |            8 |            8 pipeline stages |              0.4475 |        0.0031
    CORDIC |           12 |           12 pipeline stages |              0.0280 |        0.0000
    CORDIC |           16 |           16 pipeline stages |              0.0017 |        0.0000
       LUT |            4 |         32 entries x2 tables |              0.0207 |        0.0554
       LUT |            6 |        128 entries x2 tables |              0.0012 |        0.0031
       LUT |            8 |        512 entries x2 tables |              0.0001 |        0.0002
       LUT |           10 |       2048 entries x2 tables |              0.0000 |        0.0000
Polynomial |            2 |                 

In [ ]:
import numpy as np

def quantize(val, frac_bits, word_bits):
    """Convert a float to a signed fixed-point integer: Q(word_bits-frac_bits).frac_bits
    Clips to the representable range instead of silently wrapping - wrapping
    would corrupt the value instead of just losing precision, and real
    hardware saturating arithmetic behaves this way too."""
    scale = 2 ** frac_bits
    max_val = 2 ** (word_bits - 1) - 1
    min_val = -(2 ** (word_bits - 1))
    q = int(round(val * scale))
    return max(min_val, min(max_val, q))

def to_float(q, frac_bits):
    return q / (2 ** frac_bits)

def build_atan_table_fixed(n_iterations, angle_frac_bits, angle_word_bits):
    """atan(2^-i) in degrees, quantized to fixed-point - this is the ROM
    content in real hardware, computed once and hardcoded."""
    table = []
    for i in range(n_iterations):
        deg = np.degrees(np.arctan(2.0 ** -i))
        table.append(quantize(deg, angle_frac_bits, angle_word_bits))
    return table

def cordic_gain_recip_fixed(n_iterations, frac_bits, word_bits):
    """1/gain as a fixed-point constant, applied as a single multiply at
    the end - this is how real CORDIC implementations handle the gain
    correction (one multiply, not a division)."""
    gain = 1.0
    for i in range(n_iterations):
        gain *= np.sqrt(1 + 2.0 ** (-2 * i))
    return quantize(1.0 / gain, frac_bits, word_bits)

def cordic_vectoring_fixed(x_float, y_float, n_iterations,
                            word_bits, frac_bits,
                            angle_word_bits, angle_frac_bits,
                            atan_table_q, gain_recip_q):
    """
    Fixed-point CORDIC vectoring - integer shift/add only, matching what
    real hardware does. x,y and the angle accumulator use SEPARATE fixed-point
    formats since they represent different physical quantities (amplitude vs.
    degrees) with very different natural ranges. IMPORTANT: the angle format
    needs enough integer bits to hold up to 180 - Q8.8 (8 integer bits) only
    reaches ±128 and silently corrupts anything near 180 degrees; use at
    least 10 integer bits for degrees (Q10.6 below).
    """
    x = quantize(x_float, frac_bits, word_bits)
    y = quantize(y_float, frac_bits, word_bits)

    angle_offset_q = 0
    if x < 0:
        x, y = -x, -y
        angle_offset_q = quantize(180.0, angle_frac_bits, angle_word_bits) if y >= 0 \
            else quantize(-180.0, angle_frac_bits, angle_word_bits)

    angle_acc_q = 0
    for i in range(n_iterations):
        step_q = atan_table_q[i]
        # arithmetic right shift by i bits = multiply by 2^-i, done via
        # integer shift, exactly as hardware would - no floating point here
        x_shifted = x >> i
        y_shifted = y >> i
        if y < 0:
            x, y = x - y_shifted, y + x_shifted
            angle_acc_q -= step_q
        else:
            x, y = x + y_shifted, y - x_shifted
            angle_acc_q += step_q

    # apply gain correction as a single fixed-point multiply, then rescale
    magnitude_q = (x * gain_recip_q) >> frac_bits
    magnitude = to_float(magnitude_q, frac_bits)

    angle_q = angle_acc_q + angle_offset_q
    angle = to_float(angle_q, angle_frac_bits)
    angle = ((angle + 180) % 360) - 180

    return magnitude, angle


# === Sanity check: same known quadrant cases as before ===
word_bits, frac_bits = 16, 13          # Q3.13 for x/y
angle_word_bits, angle_frac_bits = 16, 6  # Q10.6 for angle - needs 10 integer bits to hold ±180
n_iter = 16

atan_q = build_atan_table_fixed(n_iter, angle_frac_bits, angle_word_bits)
gain_recip_q = cordic_gain_recip_fixed(n_iter, frac_bits, word_bits)

test_cases = [
    (1.0, 0.0, 1.0, 0.0),
    (0.0, 1.0, 1.0, 90.0),
    (-1.0, 0.0, 1.0, 180.0),
    (0.0, -1.0, 1.0, -90.0),
    (1.0, 1.0, np.sqrt(2), 45.0),
    (-1.0, 1.0, np.sqrt(2), 135.0),
    (-1.0, -1.0, np.sqrt(2), -135.0),
    (1.0, -1.0, np.sqrt(2), -45.0),
    (0.6, 0.8, 1.0, np.degrees(np.arctan2(0.8, 0.6))),
]
print(f"Fixed-point format: Q{word_bits-frac_bits}.{frac_bits} for x/y, "
      f"Q{angle_word_bits-angle_frac_bits}.{angle_frac_bits} for angle")
print(f"{'x':>6} {'y':>6} | {'Expected Mag':>13} {'Got Mag':>10} | "
      f"{'Expected Ang':>13} {'Got Ang':>10}")
print("-" * 70)
for x, y, exp_mag, exp_ang in test_cases:
    mag, ang = cordic_vectoring_fixed(x, y, n_iter, word_bits, frac_bits,
                                        angle_word_bits, angle_frac_bits,
                                        atan_q, gain_recip_q)
    print(f"{x:>6.2f} {y:>6.2f} | {exp_mag:>13.4f} {mag:>10.4f} | "
          f"{exp_ang:>13.2f} {ang:>10.2f}")

# === The real design decision: word width vs Phase 0's actual noise floor ===
print("\n=== Word width sweep: quantization error vs frac_bits ===")
print("(Phase 0 Run 2 reference: inherent statistical error floor was")
print(" ~0.14% amp / 0.04 deg phase at 20dB SNR, ~8.5% amp / 7.3 deg at -40dB SNR)")
np.random.seed(0)
n_test = 2000
test_angles = np.random.uniform(-180, 180, n_test)
test_r = np.random.uniform(0.1, 2, n_test)
xs = test_r * np.cos(np.radians(test_angles))
ys = test_r * np.sin(np.radians(test_angles))

print(f"\n{'frac_bits':>10} | {'word_bits':>10} | {'Max mag err %':>13} | {'Max angle err (deg)':>19}")
print("-" * 65)
for fb in [6, 8, 10, 12, 13, 14, 16]:
    wb = fb + 3  # 3 integer bits: sign + range up to ~4x
    awb, afb = 16, 6
    atan_q2 = build_atan_table_fixed(n_iter, afb, awb)
    gain_q2 = cordic_gain_recip_fixed(n_iter, fb, wb)
    max_mag_err, max_ang_err = 0, 0
    for x, y, true_r, true_ang in zip(xs, ys, test_r, test_angles):
        mag, ang = cordic_vectoring_fixed(x, y, n_iter, wb, fb, awb, afb, atan_q2, gain_q2)
        mag_err = abs(mag - true_r) / true_r * 100
        ang_err = abs(((ang - true_ang + 180) % 360) - 180)
        max_mag_err = max(max_mag_err, mag_err)
        max_ang_err = max(max_ang_err, ang_err)
    print(f"{fb:>10} | {wb:>10} | {max_mag_err:>13.4f} | {max_ang_err:>19.4f}")

print("\nGuidance: at -40dB SNR the inherent noise floor is ~8.5% amp / 7.3 deg")
print("phase error - quantization error becomes irrelevant (negligible vs")
print("noise) once it's roughly 10x smaller than that, i.e. well under 1%")
print("amp / 0.7 deg. Read off the smallest frac_bits that clears that bar -")
print("more bits past that point cost hardware resources for no real gain.")

# === Cross-check against Phase 0's actual signal chain (needs Cell 1 run first) ===
print("\n=== Cross-check against Phase 0's I/Q lock-in output ===")
from scipy.signal import butter, sosfiltfilt

np.random.seed(42)
fs, f_ref = 200_000, 1_000
true_amp, true_phase, snr_db, duration = 1.0, 37.0, 0, 0.5
lpf_cutoff = 20

t, noisy = make_test_signal(fs, f_ref, true_amp, true_phase, snr_db, duration)
ref_i = np.cos(2 * np.pi * f_ref * t)
ref_q = -np.sin(2 * np.pi * f_ref * t)
mixed_i = noisy * ref_i
mixed_q = noisy * ref_q
sos = butter(4, lpf_cutoff / (fs / 2), btype="low", output="sos")
i_filt = sosfiltfilt(sos, mixed_i)
q_filt = sosfiltfilt(sos, mixed_q)

mags, angs = [], []
for xi, yi in zip(i_filt, q_filt):
    m, a = cordic_vectoring_fixed(xi, yi, n_iter, word_bits, frac_bits,
                                    angle_word_bits, angle_frac_bits,
                                    atan_q, gain_recip_q)
    mags.append(m)
    angs.append(a)
mags = np.array(mags) * 2
angs = np.array(angs)

tail = int(len(mags) * 0.2)
print(f"Fixed-point CORDIC recovery -> Amplitude: {np.mean(mags[-tail:]):.4f} "
      f"(true {true_amp}) | Phase: {np.mean(angs[-tail:]):.2f} deg (true {true_phase})")

mag_ref, ang_ref = ideal_iq_lockin(noisy, fs, f_ref, lpf_cutoff)
print(f"Floating-point reference   -> Amplitude: {steady_state_estimate(mag_ref):.4f} "
      f"| Phase: {steady_state_estimate(ang_ref):.2f} deg")

Fixed-point format: Q3.13 for x/y, Q10.6 for angle
     x      y |  Expected Mag    Got Mag |  Expected Ang    Got Ang
----------------------------------------------------------------------
  1.00   0.00 |        1.0000     1.0005 |          0.00       0.00
  0.00   1.00 |        1.0000     1.0002 |         90.00      90.00
 -1.00   0.00 |        1.0000     1.0005 |        180.00    -180.00
  0.00  -1.00 |        1.0000     1.0005 |        -90.00     -90.00
  1.00   1.00 |        1.4142     1.4144 |         45.00      45.00
 -1.00   1.00 |        1.4142     1.4144 |        135.00     135.00
 -1.00  -1.00 |        1.4142     1.4144 |       -135.00    -135.00
  1.00  -1.00 |        1.4142     1.4144 |        -45.00     -45.00
  0.60   0.80 |        1.0000     1.0002 |         53.13      53.12

=== Word width sweep: quantization error vs frac_bits ===
(Phase 0 Run 2 reference: inherent statistical error floor was
 ~0.14% amp / 0.04 deg phase at 20dB SNR, ~8.5% amp / 7.3 deg at -40dB SNR)


In [ ]:
import numpy as np

def quantize(val, frac_bits, word_bits):
    scale = 2 ** frac_bits
    max_val = 2 ** (word_bits - 1) - 1
    min_val = -(2 ** (word_bits - 1))
    q = int(round(val * scale))
    return max(min_val, min(max_val, q))

def to_float(q, frac_bits):
    return q / (2 ** frac_bits)

def build_atan_lut_fixed(table_bits, angle_word_bits, angle_frac_bits):
    """atan(r) for r in [0,1], stored as fixed-point degrees.
    Same Q10.6 angle format as fixed-point CORDIC, for consistency."""
    size = 2 ** table_bits
    r_grid = np.linspace(0, 1, size)
    table_q = [quantize(np.degrees(np.arctan(r)), angle_frac_bits, angle_word_bits) for r in r_grid]
    return table_q

def build_mag_scale_lut_fixed(table_bits, scale_word_bits, scale_frac_bits):
    """sqrt(1+r^2) for the same r grid, stored fixed-point. Range is
    [1, sqrt(2)] so needs at least 1 integer bit above the sign bit."""
    size = 2 ** table_bits
    r_grid = np.linspace(0, 1, size)
    table_q = [quantize(np.sqrt(1 + r ** 2), scale_frac_bits, scale_word_bits) for r in r_grid]
    return table_q

def r_to_index_and_frac(r_q, r_frac_bits, table_bits, table_size):
    """Convert a fixed-point r value directly to a table index + fixed-point
    interpolation fraction, using only integer arithmetic (shifts/multiplies) -
    no floating point division, matching what real hardware would do."""
    idx_scaled = r_q * (table_size - 1)
    idx0 = idx_scaled >> r_frac_bits
    idx0 = max(0, min(table_size - 2, idx0))
    frac_q = idx_scaled - (idx0 << r_frac_bits)
    return idx0, frac_q, r_frac_bits

def lut_interp_fixed(idx0, frac_q, frac_bits_of_frac, table_q, out_frac_bits):
    """Fixed-point linear interpolation: table[idx0] + frac*(table[idx1]-table[idx0]),
    all via integer multiply + shift, no floats."""
    v0, v1 = table_q[idx0], table_q[idx0 + 1]
    diff = v1 - v0
    interp_term = (diff * frac_q) >> frac_bits_of_frac
    return v0 + interp_term

def lut_vectoring_fixed(x_float, y_float,
                          word_bits, frac_bits,           # x/y format
                          r_word_bits, r_frac_bits,        # r format
                          table_bits,
                          angle_word_bits, angle_frac_bits,
                          scale_word_bits, scale_frac_bits,
                          atan_table_q, mag_table_q):
    x_q = quantize(x_float, frac_bits, word_bits)
    y_q = quantize(y_float, frac_bits, word_bits)
    ax_q, ay_q = abs(x_q), abs(y_q)

    if ax_q == 0 and ay_q == 0:
        return 0.0, 0.0

    table_size = 2 ** table_bits
    if ax_q >= ay_q:
        r_q = (ay_q << r_frac_bits) // ax_q if ax_q != 0 else 0
        base_from_x = True
    else:
        r_q = (ax_q << r_frac_bits) // ay_q
        base_from_x = False
    r_q = min(r_q, (1 << r_frac_bits))  # clamp r to <= 1.0

    idx0, frac_q, fb = r_to_index_and_frac(r_q, r_frac_bits, table_bits, table_size)
    angle_lookup_q = lut_interp_fixed(idx0, frac_q, fb, atan_table_q, angle_frac_bits)
    scale_lookup_q = lut_interp_fixed(idx0, frac_q, fb, mag_table_q, scale_frac_bits)

    max_ax_ay_q = max(ax_q, ay_q)
    mag_q = (max_ax_ay_q * scale_lookup_q) >> scale_frac_bits
    magnitude = to_float(mag_q, frac_bits)

    angle_lookup = to_float(angle_lookup_q, angle_frac_bits)
    if base_from_x:
        angle = angle_lookup
    else:
        angle = 90 - angle_lookup

    if x_float >= 0 and y_float >= 0:
        pass
    elif x_float < 0 and y_float >= 0:
        angle = 180 - angle
    elif x_float < 0 and y_float < 0:
        angle = -(180 - angle)
    else:
        angle = -angle

    return magnitude, angle


# === Sanity check: same known quadrant cases as before ===
word_bits, frac_bits = 16, 13
r_word_bits, r_frac_bits = 16, 14
angle_word_bits, angle_frac_bits = 16, 6
scale_word_bits, scale_frac_bits = 16, 13
table_bits = 8

atan_q = build_atan_lut_fixed(table_bits, angle_word_bits, angle_frac_bits)
mag_q = build_mag_scale_lut_fixed(table_bits, scale_word_bits, scale_frac_bits)

test_cases = [
    (1.0, 0.0, 1.0, 0.0),
    (0.0, 1.0, 1.0, 90.0),
    (-1.0, 0.0, 1.0, 180.0),
    (0.0, -1.0, 1.0, -90.0),
    (1.0, 1.0, np.sqrt(2), 45.0),
    (-1.0, 1.0, np.sqrt(2), 135.0),
    (-1.0, -1.0, np.sqrt(2), -135.0),
    (1.0, -1.0, np.sqrt(2), -45.0),
    (0.6, 0.8, 1.0, np.degrees(np.arctan2(0.8, 0.6))),
]
print(f"{'x':>6} {'y':>6} | {'Expected Mag':>13} {'Got Mag':>10} | "
      f"{'Expected Ang':>13} {'Got Ang':>10}")
print("-" * 70)
for x, y, exp_mag, exp_ang in test_cases:
    mag, ang = lut_vectoring_fixed(x, y, word_bits, frac_bits, r_word_bits, r_frac_bits,
                                     table_bits, angle_word_bits, angle_frac_bits,
                                     scale_word_bits, scale_frac_bits, atan_q, mag_q)
    print(f"{x:>6.2f} {y:>6.2f} | {exp_mag:>13.4f} {mag:>10.4f} | "
          f"{exp_ang:>13.2f} {ang:>10.2f}")

# === Table bits vs word width vs accuracy - the real design decision ===
print("\n=== Table bits vs word width vs accuracy ===")
np.random.seed(0)
n_test = 2000
test_angles = np.random.uniform(-180, 180, n_test)
test_r = np.random.uniform(0.1, 2, n_test)
xs = test_r * np.cos(np.radians(test_angles))
ys = test_r * np.sin(np.radians(test_angles))

print(f"{'table_bits':>10} | {'frac_bits':>10} | {'Max mag err %':>13} | {'Max angle err (deg)':>19}")
print("-" * 65)
for tb, fb in [(4, 10), (6, 12), (8, 13), (10, 14), (12, 15)]:
    wb = fb + 3
    atan_q2 = build_atan_lut_fixed(tb, angle_word_bits, angle_frac_bits)
    mag_q2 = build_mag_scale_lut_fixed(tb, wb, fb)
    max_mag_err, max_ang_err = 0, 0
    for x, y, true_r, true_ang in zip(xs, ys, test_r, test_angles):
        mag, ang = lut_vectoring_fixed(x, y, wb, fb, r_word_bits, r_frac_bits, tb,
                                         angle_word_bits, angle_frac_bits, wb, fb,
                                         atan_q2, mag_q2)
        mag_err = abs(mag - true_r) / true_r * 100
        ang_err = abs(((ang - true_ang + 180) % 360) - 180)
        max_mag_err = max(max_mag_err, mag_err)
        max_ang_err = max(max_ang_err, ang_err)
    print(f"{tb:>10} | {fb:>10} | {max_mag_err:>13.4f} | {max_ang_err:>19.4f}")

# === Cross-check against Phase 0's I/Q lock-in output (needs Cell 1 run first) ===
print("\n=== Cross-check against Phase 0's I/Q lock-in output ===")
from scipy.signal import butter, sosfiltfilt

np.random.seed(42)
fs, f_ref = 200_000, 1_000
true_amp, true_phase, snr_db, duration = 1.0, 37.0, 0, 0.5
lpf_cutoff = 20

t, noisy = make_test_signal(fs, f_ref, true_amp, true_phase, snr_db, duration)
ref_i = np.cos(2 * np.pi * f_ref * t)
ref_q = -np.sin(2 * np.pi * f_ref * t)
mixed_i = noisy * ref_i
mixed_q = noisy * ref_q
sos = butter(4, lpf_cutoff / (fs / 2), btype="low", output="sos")
i_filt = sosfiltfilt(sos, mixed_i)
q_filt = sosfiltfilt(sos, mixed_q)

mags, angs = [], []
for xi, yi in zip(i_filt, q_filt):
    m, a = lut_vectoring_fixed(xi, yi, word_bits, frac_bits, r_word_bits, r_frac_bits, table_bits,
                                 angle_word_bits, angle_frac_bits, scale_word_bits, scale_frac_bits,
                                 atan_q, mag_q)
    mags.append(m)
    angs.append(a)
mags = np.array(mags) * 2
angs = np.array(angs)

tail = int(len(mags) * 0.2)
print(f"Fixed-point LUT recovery -> Amplitude: {np.mean(mags[-tail:]):.4f} "
      f"(true {true_amp}) | Phase: {np.mean(angs[-tail:]):.2f} deg (true {true_phase})")

mag_ref, ang_ref = ideal_iq_lockin(noisy, fs, f_ref, lpf_cutoff)
print(f"Floating-point reference -> Amplitude: {steady_state_estimate(mag_ref):.4f} "
      f"| Phase: {steady_state_estimate(ang_ref):.2f} deg")

     x      y |  Expected Mag    Got Mag |  Expected Ang    Got Ang
----------------------------------------------------------------------
  1.00   0.00 |        1.0000     1.0000 |          0.00       0.00
  0.00   1.00 |        1.0000     1.0000 |         90.00      90.00
 -1.00   0.00 |        1.0000     1.0000 |        180.00     180.00
  0.00  -1.00 |        1.0000     1.0000 |        -90.00     -90.00
  1.00   1.00 |        1.4142     1.4142 |         45.00      45.00
 -1.00   1.00 |        1.4142     1.4142 |        135.00     135.00
 -1.00  -1.00 |        1.4142     1.4142 |       -135.00    -135.00
  1.00  -1.00 |        1.4142     1.4142 |        -45.00     -45.00
  0.60   0.80 |        1.0000     0.9999 |         53.13      53.14

=== Table bits vs word width vs accuracy ===
table_bits |  frac_bits | Max mag err % | Max angle err (deg)
-----------------------------------------------------------------
         4 |         10 |        1.1463 |              0.3435
         6 | 

In [ ]:
import numpy as np

def quantize(val, frac_bits, word_bits):
    scale = 2 ** frac_bits
    max_val = 2 ** (word_bits - 1) - 1
    min_val = -(2 ** (word_bits - 1))
    q = int(round(val * scale))
    return max(min_val, min(max_val, q))

def to_float(q, frac_bits):
    return q / (2 ** frac_bits)

def fit_atan_poly_power(order):
    """Chebyshev fit for numerical stability, converted to plain power-series
    coefficients (highest degree first) - real hardware evaluates a plain
    polynomial via Horner's method, Chebyshev basis is just a fitting trick."""
    r_fit = np.linspace(0, 1, 500)
    atan_vals = np.degrees(np.arctan(r_fit))
    cheb_c = np.polynomial.chebyshev.chebfit(r_fit, atan_vals, order)
    power_c = np.polynomial.chebyshev.cheb2poly(cheb_c)  # increasing degree
    return power_c[::-1]  # highest degree first, for Horner

def fit_mag_poly_power(order):
    r_fit = np.linspace(0, 1, 500)
    scale_vals = np.sqrt(1 + r_fit ** 2)
    cheb_c = np.polynomial.chebyshev.chebfit(r_fit, scale_vals, order)
    power_c = np.polynomial.chebyshev.cheb2poly(cheb_c)
    return power_c[::-1]

def quantize_coeffs(coeffs, frac_bits, word_bits):
    return [quantize(c, frac_bits, word_bits) for c in coeffs]

def poly_eval_fixed_horner(r_q, r_frac_bits, coeffs_q, coeff_frac_bits, coeff_word_bits):
    """Horner's method in fixed point: acc = ((acc * r) >> r_frac_bits) + next_coeff,
    all integer arithmetic. acc stays in the SAME fixed-point format as the
    coefficients throughout - this is what makes the shift amount consistent."""
    acc = coeffs_q[0]
    max_val = 2 ** (coeff_word_bits - 1) - 1
    min_val = -(2 ** (coeff_word_bits - 1))
    for c in coeffs_q[1:]:
        acc = ((acc * r_q) >> r_frac_bits) + c
        acc = max(min_val, min(max_val, acc))  # saturate, matching real hardware
    return acc

def poly_vectoring_fixed(x_float, y_float,
                           word_bits, frac_bits,          # x/y format
                           r_word_bits, r_frac_bits,       # r format
                           atan_coeffs_q, atan_frac_bits, atan_word_bits,
                           mag_coeffs_q, mag_frac_bits, mag_word_bits):
    x_q = quantize(x_float, frac_bits, word_bits)
    y_q = quantize(y_float, frac_bits, word_bits)
    ax_q, ay_q = abs(x_q), abs(y_q)

    if ax_q == 0 and ay_q == 0:
        return 0.0, 0.0

    if ax_q >= ay_q:
        r_q = (ay_q << r_frac_bits) // ax_q if ax_q != 0 else 0
        base_from_x = True
    else:
        r_q = (ax_q << r_frac_bits) // ay_q
        base_from_x = False
    r_q = min(r_q, (1 << r_frac_bits))

    angle_lookup_q = poly_eval_fixed_horner(r_q, r_frac_bits, atan_coeffs_q, atan_frac_bits, atan_word_bits)
    scale_q = poly_eval_fixed_horner(r_q, r_frac_bits, mag_coeffs_q, mag_frac_bits, mag_word_bits)

    max_ax_ay_q = max(ax_q, ay_q)
    mag_q = (max_ax_ay_q * scale_q) >> mag_frac_bits
    magnitude = to_float(mag_q, frac_bits)

    angle_lookup = to_float(angle_lookup_q, atan_frac_bits)
    if base_from_x:
        angle = angle_lookup
    else:
        angle = 90 - angle_lookup

    if x_float >= 0 and y_float >= 0:
        pass
    elif x_float < 0 and y_float >= 0:
        angle = 180 - angle
    elif x_float < 0 and y_float < 0:
        angle = -(180 - angle)
    else:
        angle = -angle

    return magnitude, angle


# === Sanity check: same known quadrant cases as before ===
word_bits, frac_bits = 16, 13
r_word_bits, r_frac_bits = 16, 14
atan_word_bits, atan_frac_bits = 16, 8   # coeffs up to ~58 -> need 7+ integer bits
mag_word_bits, mag_frac_bits = 16, 13    # coeffs all < 1.1 -> plenty of headroom

order = 5
atan_coeffs = fit_atan_poly_power(order)
mag_coeffs = fit_mag_poly_power(3)
atan_coeffs_q = quantize_coeffs(atan_coeffs, atan_frac_bits, atan_word_bits)
mag_coeffs_q = quantize_coeffs(mag_coeffs, mag_frac_bits, mag_word_bits)

test_cases = [
    (1.0, 0.0, 1.0, 0.0),
    (0.0, 1.0, 1.0, 90.0),
    (-1.0, 0.0, 1.0, 180.0),
    (0.0, -1.0, 1.0, -90.0),
    (1.0, 1.0, np.sqrt(2), 45.0),
    (-1.0, 1.0, np.sqrt(2), 135.0),
    (-1.0, -1.0, np.sqrt(2), -135.0),
    (1.0, -1.0, np.sqrt(2), -45.0),
    (0.6, 0.8, 1.0, np.degrees(np.arctan2(0.8, 0.6))),
]
print(f"{'x':>6} {'y':>6} | {'Expected Mag':>13} {'Got Mag':>10} | "
      f"{'Expected Ang':>13} {'Got Ang':>10}")
print("-" * 70)
for x, y, exp_mag, exp_ang in test_cases:
    mag, ang = poly_vectoring_fixed(x, y, word_bits, frac_bits, r_word_bits, r_frac_bits,
                                      atan_coeffs_q, atan_frac_bits, atan_word_bits,
                                      mag_coeffs_q, mag_frac_bits, mag_word_bits)
    print(f"{x:>6.2f} {y:>6.2f} | {exp_mag:>13.4f} {mag:>10.4f} | "
          f"{exp_ang:>13.2f} {ang:>10.2f}")

# === Order vs accuracy - watch for a plateau caused by coefficient precision ===
print("\n=== Polynomial order vs accuracy (fixed-point) ===")
np.random.seed(0)
n_test = 2000
test_angles = np.random.uniform(-180, 180, n_test)
test_r = np.random.uniform(0.1, 2, n_test)
xs = test_r * np.cos(np.radians(test_angles))
ys = test_r * np.sin(np.radians(test_angles))

print(f"{'order':>6} | {'Max mag err %':>13} | {'Max angle err (deg)':>19}")
print("-" * 50)
for ordr in [2, 3, 4, 5, 6]:
    atan_c = fit_atan_poly_power(ordr)
    mag_c = fit_mag_poly_power(min(ordr, 5))
    atan_c_q = quantize_coeffs(atan_c, atan_frac_bits, atan_word_bits)
    mag_c_q = quantize_coeffs(mag_c, mag_frac_bits, mag_word_bits)
    max_mag_err, max_ang_err = 0, 0
    for x, y, true_r, true_ang in zip(xs, ys, test_r, test_angles):
        mag, ang = poly_vectoring_fixed(x, y, word_bits, frac_bits, r_word_bits, r_frac_bits,
                                          atan_c_q, atan_frac_bits, atan_word_bits,
                                          mag_c_q, mag_frac_bits, mag_word_bits)
        mag_err = abs(mag - true_r) / true_r * 100
        ang_err = abs(((ang - true_ang + 180) % 360) - 180)
        max_mag_err = max(max_mag_err, mag_err)
        max_ang_err = max(max_ang_err, ang_err)
    print(f"{ordr:>6} | {max_mag_err:>13.4f} | {max_ang_err:>19.4f}")

print("\nIf error stops improving past order 3-4, that's coefficient/accumulator")
print("precision becoming the bottleneck, not polynomial order - confirmed by")
print("widening coefficient fractional bits and seeing error drop further.")
print("This is a genuinely different limiting factor than in floating-point,")
print("worth stating explicitly rather than assuming order alone drives accuracy.")

# === Cross-check against Phase 0's I/Q lock-in output (needs Cell 1 run first) ===
print("\n=== Cross-check against Phase 0's I/Q lock-in output ===")
from scipy.signal import butter, sosfiltfilt

np.random.seed(42)
fs, f_ref = 200_000, 1_000
true_amp, true_phase, snr_db, duration = 1.0, 37.0, 0, 0.5
lpf_cutoff = 20

t, noisy = make_test_signal(fs, f_ref, true_amp, true_phase, snr_db, duration)
ref_i = np.cos(2 * np.pi * f_ref * t)
ref_q = -np.sin(2 * np.pi * f_ref * t)
mixed_i = noisy * ref_i
mixed_q = noisy * ref_q
sos = butter(4, lpf_cutoff / (fs / 2), btype="low", output="sos")
i_filt = sosfiltfilt(sos, mixed_i)
q_filt = sosfiltfilt(sos, mixed_q)

mags, angs = [], []
for xi, yi in zip(i_filt, q_filt):
    m, a = poly_vectoring_fixed(xi, yi, word_bits, frac_bits, r_word_bits, r_frac_bits,
                                  atan_coeffs_q, atan_frac_bits, atan_word_bits,
                                  mag_coeffs_q, mag_frac_bits, mag_word_bits)
    mags.append(m)
    angs.append(a)
mags = np.array(mags) * 2
angs = np.array(angs)

tail = int(len(mags) * 0.2)
print(f"Fixed-point Poly recovery -> Amplitude: {np.mean(mags[-tail:]):.4f} "
      f"(true {true_amp}) | Phase: {np.mean(angs[-tail:]):.2f} deg (true {true_phase})")

mag_ref, ang_ref = ideal_iq_lockin(noisy, fs, f_ref, lpf_cutoff)
print(f"Floating-point reference  -> Amplitude: {steady_state_estimate(mag_ref):.4f} "
      f"| Phase: {steady_state_estimate(ang_ref):.2f} deg")

     x      y |  Expected Mag    Got Mag |  Expected Ang    Got Ang
----------------------------------------------------------------------
  1.00   0.00 |        1.0000     1.0002 |          0.00       0.00
  0.00   1.00 |        1.0000     1.0002 |         90.00      90.00
 -1.00   0.00 |        1.0000     1.0002 |        180.00     180.00
  0.00  -1.00 |        1.0000     1.0002 |        -90.00     -90.00
  1.00   1.00 |        1.4142     1.4141 |         45.00      45.00
 -1.00   1.00 |        1.4142     1.4141 |        135.00     135.00
 -1.00  -1.00 |        1.4142     1.4141 |       -135.00    -135.00
  1.00  -1.00 |        1.4142     1.4141 |        -45.00     -45.00
  0.60   0.80 |        1.0000     0.9998 |         53.13      53.14

=== Polynomial order vs accuracy (fixed-point) ===
 order | Max mag err % | Max angle err (deg)
--------------------------------------------------
     2 |        0.6717 |              0.3002
     3 |        0.1638 |              0.1322
     4 |   

In [ ]:
import numpy as np

def quantize(val, frac_bits, word_bits):
    scale = 2 ** frac_bits
    max_val = 2 ** (word_bits - 1) - 1
    min_val = -(2 ** (word_bits - 1))
    q = int(round(val * scale))
    return max(min_val, min(max_val, q))

def to_float(q, frac_bits):
    return q / (2 ** frac_bits)

def cic_required_bit_growth(input_bits, n_stages, decimation, diff_delay):
    """Hogenauer's classic bound on how many extra integer bits the internal
    CIC accumulators need to guarantee no overflow, independent of the
    actual data. This is a REQUIRED hardware register-sizing calculation -
    skipping it risks silent wraparound on real hardware."""
    growth_bits = int(np.ceil(n_stages * np.log2(decimation * diff_delay)))
    return input_bits + growth_bits

def cic_decimate_fixed(x_float, decimation, n_stages, diff_delay, input_frac_bits, input_word_bits):
    """Quantize input once, then run the integrator/comb chain in pure
    integer arithmetic (numpy int64 is wide enough for our bit-growth
    here - real hardware would size each stage's register per the
    Hogenauer bound above instead of using one generic wide type)."""
    x_q = np.array([quantize(v, input_frac_bits, input_word_bits) for v in x_float], dtype=np.int64)

    y = x_q.copy()
    for _ in range(n_stages):
        y = np.cumsum(y)  # pure integer addition - exactly what hardware does
    y = y[::decimation]
    for _ in range(n_stages):
        y = y - np.concatenate([np.zeros(diff_delay, dtype=np.int64), y[:-diff_delay]])

    dc_gain = decimation ** n_stages
    # NOTE: dc_gain here (100^4) is NOT a power of 2, so this rescale needs
    # an actual integer divide - real hardware would prefer a power-of-2
    # decimation factor (e.g. 128 instead of 100) specifically so this
    # becomes a cheap right-shift instead of a divider circuit.
    y_rescaled_q = y // dc_gain
    return np.array([to_float(v, input_frac_bits) for v in y_rescaled_q])

def quantize_fir_taps(taps_float, frac_bits, word_bits):
    return [quantize(t, frac_bits, word_bits) for t in taps_float]

def fir_apply_fixed(x_float, taps_q, tap_frac_bits, tap_word_bits, x_frac_bits, x_word_bits):
    """Fixed-point FIR: quantize input, integer multiply-accumulate against
    quantized taps, then rescale. This is a real MAC operation - unlike
    CIC, FIR genuinely needs multiplier hardware (DSP slices)."""
    x_q = [quantize(v, x_frac_bits, x_word_bits) for v in x_float]
    n = len(x_q)
    ntaps = len(taps_q)
    y_q = np.zeros(n, dtype=np.int64)
    for i in range(n):
        acc = 0
        for k in range(ntaps):
            if i - k >= 0:
                acc += taps_q[k] * x_q[i - k]
        y_q[i] = acc
    # product of two fixed-point values doubles the fractional bits -
    # rescale back down by the tap's fractional bits (a clean right-shift
    # since tap_frac_bits is chosen as a power-of-2-friendly value)
    y_q_rescaled = y_q >> tap_frac_bits
    return np.array([to_float(v, x_frac_bits) for v in y_q_rescaled])


# === Bit-growth analysis - a REQUIRED hardware register-sizing step ===
decimation, n_stages, diff_delay = 100, 4, 1
input_word_bits, input_frac_bits = 16, 13

req_bits = cic_required_bit_growth(input_word_bits, n_stages, decimation, diff_delay)
print("=== CIC bit-growth analysis (Hogenauer bound) ===")
print(f"Input width: {input_word_bits} bits | Stages: {n_stages} | "
      f"Decimation: {decimation} | Comb delay: {diff_delay}")
print(f"Required internal accumulator width: {req_bits} bits")
print(f"(growth of {req_bits - input_word_bits} bits beyond the input width - "
      f"using fewer bits internally risks silent overflow/wraparound)")

print(f"\nNote: decimation={decimation} is NOT a power of 2, so the final")
print(f"DC-gain rescale ({decimation}^{n_stages} = {decimation**n_stages}) needs a real")
print(f"integer divider in hardware. Choosing decimation=128 instead would")
print(f"make this a free right-shift (128^{n_stages} = 2^{7*n_stages}).")

# === Cross-check: fixed-point CIC vs floating-point reference (needs cic_decimate from your CIC/FIR cell) ===
print("\n=== Cross-check: fixed-point CIC vs floating-point reference ===")
np.random.seed(1)
test_signal = np.random.uniform(-1, 1, 500)
y_float = cic_decimate(test_signal, decimation, n_stages, diff_delay)
y_fixed = cic_decimate_fixed(test_signal, decimation, n_stages, diff_delay,
                               input_frac_bits, input_word_bits)
max_err = np.max(np.abs(y_float - y_fixed[:len(y_float)]))
print(f"Max absolute difference: {max_err:.6f}")

# === Cross-check: fixed-point FIR vs floating-point reference ===
print("\n=== Cross-check: fixed-point FIR vs floating-point reference ===")
fir_taps_float = design_compensating_fir(decimation, n_stages, diff_delay, 21, 2000)
tap_word_bits, tap_frac_bits = 16, 14  # taps are all small (<1), plenty of headroom
taps_q = quantize_fir_taps(fir_taps_float, tap_frac_bits, tap_word_bits)

from scipy.signal import lfilter
test_signal2 = np.random.uniform(-1, 1, 200)
y_float_fir = lfilter(fir_taps_float, 1.0, test_signal2)
y_fixed_fir = fir_apply_fixed(test_signal2, taps_q, tap_frac_bits, tap_word_bits,
                                input_frac_bits, input_word_bits)
max_err_fir = np.max(np.abs(y_float_fir - y_fixed_fir))
print(f"Max absolute difference: {max_err_fir:.6f}")

# === Full end-to-end chain cross-check against Phase 0's golden model ===
print("\n=== Full chain: mix -> fixed CIC -> fixed FIR, vs floating-point reference ===")
np.random.seed(42)
fs, f_ref = 200_000, 1_000
true_amp, true_phase, snr_db, duration = 1.0, 37.0, 0, 0.1

t, noisy = make_test_signal(fs, f_ref, true_amp, true_phase, snr_db, duration)
ref_i = np.cos(2 * np.pi * f_ref * t)
ref_q = -np.sin(2 * np.pi * f_ref * t)
mixed_i = noisy * ref_i
mixed_q = noisy * ref_q

i_cic_f = cic_decimate(mixed_i, decimation, n_stages, diff_delay)
q_cic_f = cic_decimate(mixed_q, decimation, n_stages, diff_delay)
i_comp_f = lfilter(fir_taps_float, 1.0, i_cic_f)
q_comp_f = lfilter(fir_taps_float, 1.0, q_cic_f)
mag_f = 2 * np.sqrt(i_comp_f ** 2 + q_comp_f ** 2)
tail_f = int(len(mag_f) * 0.3)
print(f"Floating-point chain amplitude: {np.mean(mag_f[-tail_f:]):.4f} (true {true_amp})")

i_cic_x = cic_decimate_fixed(mixed_i, decimation, n_stages, diff_delay, input_frac_bits, input_word_bits)
q_cic_x = cic_decimate_fixed(mixed_q, decimation, n_stages, diff_delay, input_frac_bits, input_word_bits)
i_comp_x = fir_apply_fixed(i_cic_x, taps_q, tap_frac_bits, tap_word_bits, input_frac_bits, input_word_bits)
q_comp_x = fir_apply_fixed(q_cic_x, taps_q, tap_frac_bits, tap_word_bits, input_frac_bits, input_word_bits)
mag_x = 2 * np.sqrt(i_comp_x ** 2 + q_comp_x ** 2)
tail_x = int(len(mag_x) * 0.3)
print(f"Fixed-point chain amplitude:    {np.mean(mag_x[-tail_x:]):.4f} (true {true_amp})")

=== CIC bit-growth analysis (Hogenauer bound) ===
Input width: 16 bits | Stages: 4 | Decimation: 100 | Comb delay: 1
Required internal accumulator width: 43 bits
(growth of 27 bits beyond the input width - using fewer bits internally risks silent overflow/wraparound)

Note: decimation=100 is NOT a power of 2, so the final
DC-gain rescale (100^4 = 100000000) needs a real
integer divider in hardware. Choosing decimation=128 instead would
make this a free right-shift (128^4 = 2^28).

=== Cross-check: fixed-point CIC vs floating-point reference ===
Max absolute difference: 0.000122

=== Cross-check: fixed-point FIR vs floating-point reference ===
Max absolute difference: 0.000237

=== Full chain: mix -> fixed CIC -> fixed FIR, vs floating-point reference ===
Floating-point chain amplitude: 0.9942 (true 1.0)
Fixed-point chain amplitude:    0.9939 (true 1.0)


In [ ]:
# =============================================================================
# PHASE 3 / STEP 1: Golden test-vector export for CORDIC, LUT, polynomial
#
# Needs these earlier cells already run in THIS session (your Phase 2 cells):
#   fixed CORDIC cell : quantize, build_atan_table_fixed, cordic_gain_recip_fixed,
#                       cordic_vectoring_fixed
#   fixed LUT cell    : build_atan_lut_fixed, build_mag_scale_lut_fixed,
#                       lut_vectoring_fixed
#   fixed poly cell   : fit_atan_poly_power, fit_mag_poly_power, quantize_coeffs,
#                       poly_vectoring_fixed
#
# What it does:
#   1. Builds ONE shared stimulus set (corners + exhaustive tiny window + LUT
#      boundary hits + 4 random families), as raw integers (x_q, y_q).
#   2. Runs an INTEGER-domain golden model of each method (what the hardware
#      sees: ints in, ints out) and asserts it matches your validated Phase 2
#      float-returning functions EXACTLY on every vector.
#   3. Checks each method against an independent float64 reference
#      (numpy arctan2/hypot), bucketed by input magnitude.
#   4. Reports the register widths the hardware actually needs.
#   5. Writes per-method CSVs + a C++ header with the exact ROM contents.
# =============================================================================
import os
import numpy as np

# ---- Config: must equal the validated Phase 2 configs ----------------------
W, F = 16, 13                       # x/y: Q3.13
N_ITER, AW, AF = 16, 16, 6          # CORDIC: iterations, angle Q10.6
R_W, R_F = 16, 14                   # r = min/max ratio, Q2.14
TABLE_BITS = 8                      # LUT: 256 entries
SC_W, SC_F = 16, 13                 # sqrt(1+r^2) scale, Q3.13
LUT_AW, LUT_AF = 16, 6              # LUT angle Q10.6
P_AW, P_AF = 16, 8                  # poly atan coeff/acc format
P_MW, P_MF = 16, 13                 # poly mag-scale coeff/acc format
ATAN_ORDER, MAG_ORDER = 5, 3
SEED = 1234

XMIN, XMAX = -(1 << (W - 1)), (1 << (W - 1)) - 1

# ---- Output folder ----------------------------------------------------------
if os.path.ismount('/content/drive'):
    OUT_DIR = '/content/drive/MyDrive/lockin_project/golden_vectors'
else:
    OUT_DIR = '/content/golden_vectors'
    print("NOTE: Drive not mounted - writing to local /content (lost on runtime reset)")
os.makedirs(OUT_DIR, exist_ok=True)

# ---- ROM contents, generated by YOUR validated builders ---------------------
atan_rom = build_atan_table_fixed(N_ITER, AF, AW)
gain_recip = cordic_gain_recip_fixed(N_ITER, F, W)
lut_atan = build_atan_lut_fixed(TABLE_BITS, LUT_AW, LUT_AF)
lut_mag = build_mag_scale_lut_fixed(TABLE_BITS, SC_W, SC_F)
p_atan = quantize_coeffs(fit_atan_poly_power(ATAN_ORDER), P_AF, P_AW)
p_mag = quantize_coeffs(fit_mag_poly_power(MAG_ORDER), P_MF, P_MW)

# =============================================================================
# 1. Stimulus set (integers only)
# =============================================================================
def build_stimulus():
    seen, out = set(), []
    def add(pairs):
        for x, y in pairs:
            x, y = int(max(XMIN, min(XMAX, x))), int(max(XMIN, min(XMAX, y)))
            if (x, y) not in seen:
                seen.add((x, y)); out.append((x, y))
    # (a) all combinations of edge values, incl. (0,0), min/max representable
    edge = [XMIN, XMIN + 1, -8192, -1, 0, 1, 8192, XMAX - 1, XMAX]
    add([(x, y) for x in edge for y in edge])
    # (b) pure axes and 45/135/225/315 diagonals at many magnitudes
    for m in [1, 2, 3, 7, 64, 1000, 8192, 16384, 23170, XMAX]:
        add([(m, m), (-m, m), (-m, -m), (m, -m), (m, 0), (-m, 0), (0, m), (0, -m)])
    n_corner = len(out)                      # the "corner" file = (a) + (b)
    # (c) LUT boundary hits: with the larger operand = 16384, r_q equals the
    #     smaller operand exactly, so every index boundary can be landed on.
    size = 1 << TABLE_BITS
    for k in range(size):
        base_rq = (k * (1 << R_F)) // (size - 1)
        for rq in (base_rq - 1, base_rq, base_rq + 1):
            if 0 <= rq <= 16384:
                add([(16384, rq), (rq, 16384)])
    # (d) exhaustive tiny window (LSB-level behaviour near zero)
    add([(x, y) for x in range(-24, 25) for y in range(-24, 25)])
    # (e) random families (fixed seed -> reproducible)
    rng = np.random.default_rng(SEED)
    add([tuple(v) for v in rng.integers(XMIN, XMAX + 1, size=(8000, 2))])         # full range
    ang = rng.uniform(-np.pi, np.pi, 8000); r = rng.uniform(0.05, 2.0, 8000)      # notebook-style
    add([(int(round(a)), int(round(b))) for a, b in
         zip(r * np.cos(ang) * 2**F, r * np.sin(ang) * 2**F)])
    add([tuple(v) for v in rng.integers(-256, 257, size=(4000, 2))])              # low amplitude
    ang = rng.uniform(-np.pi, np.pi, 4000); r = 10 ** rng.uniform(0, 4.6, 4000)   # log-uniform radius
    add([(int(round(a)), int(round(b))) for a, b in
         zip(r * np.cos(ang), r * np.sin(ang))])
    return out, n_corner

stim, n_corner = build_stimulus()
print(f"Stimulus vectors: {len(stim)} total ({n_corner} in the corner-case subset)")

# =============================================================================
# 2. Integer-domain golden models (ints in, ints out - what the HW sees)
# =============================================================================
A180_C = 180 << AF          # CORDIC angle units
A90_L, A180_L = 90 << LUT_AF, 180 << LUT_AF
A90_P, A180_P = 90 << P_AF, 180 << P_AF

def cordic_int(xq, yq, track=None):
    x, y, off = xq, yq, 0
    if x < 0:
        x, y = -x, -y
        off = A180_C if y >= 0 else -A180_C
    acc = 0
    for i in range(N_ITER):
        xs, ys = x >> i, y >> i
        if y < 0:
            x, y = x - ys, y + xs; acc -= atan_rom[i]
        else:
            x, y = x + ys, y - xs; acc += atan_rom[i]
        if track is not None:
            track['xy'] = max(track['xy'], abs(x), abs(y))
    if track is not None:
        track['acc'] = max(track['acc'], abs(acc + off))
    mag_q = (x * gain_recip) >> F
    a = ((acc + off + A180_C) % (2 * A180_C)) - A180_C     # wrap to [-180, 180)
    return mag_q, a

def _ratio(ax, ay):
    if ax >= ay:
        return (ay << R_F) // ax, True
    return (ax << R_F) // ay, False

def _fold(base_from_x, lookup, a90, a180, xq, yq):
    a = lookup if base_from_x else a90 - lookup
    if xq >= 0 and yq >= 0: return a
    if xq < 0 and yq >= 0:  return a180 - a
    if xq < 0 and yq < 0:   return -(a180 - a)
    return -a

def lut_int(xq, yq):
    ax, ay = abs(xq), abs(yq)
    if ax == 0 and ay == 0:
        return 0, 0
    rq, bfx = _ratio(ax, ay)
    rq = min(rq, 1 << R_F)
    size = 1 << TABLE_BITS
    scaled = rq * (size - 1)
    i0 = max(0, min(size - 2, scaled >> R_F))
    fr = scaled - (i0 << R_F)
    look_a = lut_atan[i0] + (((lut_atan[i0 + 1] - lut_atan[i0]) * fr) >> R_F)
    look_s = lut_mag[i0] + (((lut_mag[i0 + 1] - lut_mag[i0]) * fr) >> R_F)
    mag_q = (max(ax, ay) * look_s) >> SC_F
    return mag_q, _fold(bfx, look_a, A90_L, A180_L, xq, yq)

def _horner(rq, coeffs, wbits):
    hi, lo = (1 << (wbits - 1)) - 1, -(1 << (wbits - 1))
    acc = coeffs[0]
    for c in coeffs[1:]:
        acc = max(lo, min(hi, ((acc * rq) >> R_F) + c))
    return acc

def poly_int(xq, yq):
    ax, ay = abs(xq), abs(yq)
    if ax == 0 and ay == 0:
        return 0, 0
    rq, bfx = _ratio(ax, ay)
    rq = min(rq, 1 << R_F)
    look_a = _horner(rq, p_atan, P_AW)
    scale = _horner(rq, p_mag, P_MW)
    mag_q = (max(ax, ay) * scale) >> P_MF
    return mag_q, _fold(bfx, look_a, A90_P, A180_P, xq, yq)

# per method: (int model, angle frac bits, notebook float-returning function)
def nb_cordic(x, y):
    return cordic_vectoring_fixed(x, y, N_ITER, W, F, AW, AF, atan_rom, gain_recip)
def nb_lut(x, y):
    return lut_vectoring_fixed(x, y, W, F, R_W, R_F, TABLE_BITS, LUT_AW, LUT_AF,
                               SC_W, SC_F, lut_atan, lut_mag)
def nb_poly(x, y):
    return poly_vectoring_fixed(x, y, W, F, R_W, R_F, p_atan, P_AF, P_AW, p_mag, P_MF, P_MW)

METHODS = {
    'cordic': (cordic_int, AF, nb_cordic),
    'lut':    (lut_int,    LUT_AF, nb_lut),
    'poly':   (poly_int,   P_AF, nb_poly),
}

# =============================================================================
# 3. Run models, assert exact match with Phase 2 code, compare to float64 ref
# =============================================================================
def signed_bits(lo, hi):
    return max(int(hi).bit_length() + 1, int(-lo - 1).bit_length() + 1 if lo < 0 else 1)

def wrap_err(a_deg, b_deg):
    return abs(((a_deg - b_deg + 180.0) % 360.0) - 180.0)

xs = np.array([v[0] for v in stim], dtype=np.int64)
ys = np.array([v[1] for v in stim], dtype=np.int64)
true_mag_lsb = np.hypot(xs, ys)                       # in units of 2^-F (input LSB)
true_ang = np.degrees(np.arctan2(ys, xs))
is_zero = (xs == 0) & (ys == 0)
mag_bucket = np.where(true_mag_lsb < 16, 0, np.where(true_mag_lsb < 256, 1, 2))
bucket_names = ['|v|<16 LSB', '16-256 LSB', '>=256 LSB']

results = {}
for name, (fint, afrac, fnb) in METHODS.items():
    track = {'xy': 0, 'acc': 0} if name == 'cordic' else None
    mags = np.empty(len(stim), dtype=np.int64)
    angs = np.empty(len(stim), dtype=np.int64)
    for i, (x, y) in enumerate(stim):
        m, a = (fint(x, y, track) if track is not None else fint(x, y))
        mags[i], angs[i] = m, a
        # exact agreement with the validated Phase 2 function
        m_nb, a_nb = fnb(x / 2**F, y / 2**F)
        assert m_nb * 2**F == m, f"{name}: magnitude mismatch at {(x, y)}: {m_nb*2**F} vs {m}"
        assert a_nb * 2**afrac == a, f"{name}: angle mismatch at {(x, y)}: {a_nb*2**afrac} vs {a}"
    results[name] = (mags, angs, track)
print(f"\nInteger golden models match Phase 2 functions EXACTLY on all {len(stim)} vectors "
      f"(cordic, lut, poly).")

print("\n=== Independent float64 reference (numpy arctan2/hypot on the same integer inputs) ===")
print(f"{'method':>7} | {'bucket':>12} | {'n':>6} | {'max |mag err| (LSB)':>19} | "
      f"{'max mag err %':>13} | {'max ang err (deg)':>17}")
print("-" * 92)
for name, (fint, afrac, _) in METHODS.items():
    mags, angs, _ = results[name]
    ang_deg = angs / 2**afrac
    for b in range(3):
        sel = (mag_bucket == b) & ~is_zero
        if not sel.any():
            continue
        merr = np.abs(mags[sel] - true_mag_lsb[sel])
        aerr = np.array([wrap_err(a, t) for a, t in zip(ang_deg[sel], true_ang[sel])])
        print(f"{name:>7} | {bucket_names[b]:>12} | {sel.sum():>6} | {merr.max():>19.2f} | "
              f"{(merr / true_mag_lsb[sel]).max() * 100:>13.4f} | {aerr.max():>17.4f}")

# =============================================================================
# 4. Behaviour at special inputs + hardware register widths
# =============================================================================
print("\n=== Special-input behaviour of the golden models ===")
for name, (fint, afrac, _) in METHODS.items():
    m0, a0 = fint(0, 0)
    mneg, aneg = fint(-1000, 0)
    print(f"{name:>7}: (0,0) -> mag {m0}, angle {a0/2**afrac:.3f} deg | "
          f"(-1000,0) -> angle {aneg/2**afrac:.3f} deg")

print("\n=== Register widths the hardware needs (from observed extremes) ===")
tr = results['cordic'][2]
print(f"CORDIC x/y datapath inside the loop: max |value| {tr['xy']} -> "
      f"{signed_bits(-tr['xy'], tr['xy'])} signed bits (input word is {W})")
print(f"CORDIC angle accumulator + offset:   max |value| {tr['acc']} -> "
      f"{signed_bits(-tr['acc'], tr['acc'])} signed bits (angle word is {AW})")
for name in METHODS:
    mags, angs, _ = results[name]
    print(f"{name:>7} magnitude out: [{mags.min()}, {mags.max()}] -> "
          f"{signed_bits(mags.min(), mags.max())} signed / "
          f"{int(mags.max()).bit_length()} unsigned bits (Q3.13 word is {W} signed)")
    print(f"{name:>7} angle out:     [{angs.min()}, {angs.max()}] -> "
          f"{signed_bits(angs.min(), angs.max())} signed bits")

# =============================================================================
# 5. Write CSVs + C++ ROM header
# =============================================================================
def write_csv(path, rows, header):
    with open(path, 'w') as f:
        f.write(header + "\n")
        for r in rows:
            f.write(",".join(str(int(v)) for v in r) + "\n")

for name, (fint, afrac, _) in METHODS.items():
    mags, angs, _ = results[name]
    # angle is a don't-care only where the golden model itself is undefined
    chk = np.where(is_zero & (name == 'cordic'), 0, 1)
    rows_full = list(zip(xs, ys, mags, angs, chk))
    write_csv(f"{OUT_DIR}/{name}_full.csv", rows_full, "x_q,y_q,mag_q,ang_q,ang_check")
    write_csv(f"{OUT_DIR}/{name}_corner.csv", rows_full[:n_corner],
              "x_q,y_q,mag_q,ang_q,ang_check")

def cpp_array(ctype, name, vals):
    body = ",".join(str(int(v)) for v in vals)
    return f"static const {ctype} {name}[{len(vals)}] = {{{body}}};\n"

with open(f"{OUT_DIR}/lockin_golden_params.h", 'w') as f:
    f.write("// Auto-generated from the validated Phase 2 Python model. Do not hand-edit.\n")
    f.write("#pragma once\n")
    f.write(f"#define XY_WORD {W}\n#define XY_FRAC {F}\n")
    f.write(f"#define CORDIC_N_ITER {N_ITER}\n#define CORDIC_ANG_WORD {AW}\n#define CORDIC_ANG_FRAC {AF}\n")
    f.write(f"#define CORDIC_GAIN_RECIP {gain_recip}\n")
    f.write(cpp_array("int", "CORDIC_ATAN_ROM", atan_rom))
    f.write(f"#define R_FRAC {R_F}\n#define LUT_TABLE_BITS {TABLE_BITS}\n")
    f.write(f"#define LUT_ANG_FRAC {LUT_AF}\n#define LUT_SCALE_FRAC {SC_F}\n")
    f.write(cpp_array("int", "LUT_ATAN_ROM", lut_atan))
    f.write(cpp_array("int", "LUT_MAG_ROM", lut_mag))
    f.write(f"#define POLY_ATAN_ORDER {ATAN_ORDER}\n#define POLY_ATAN_FRAC {P_AF}\n#define POLY_ATAN_WORD {P_AW}\n")
    f.write(f"#define POLY_MAG_ORDER {MAG_ORDER}\n#define POLY_MAG_FRAC {P_MF}\n#define POLY_MAG_WORD {P_MW}\n")
    f.write("// coefficients: highest degree first (Horner order)\n")
    f.write(cpp_array("int", "POLY_ATAN_COEFFS", p_atan))
    f.write(cpp_array("int", "POLY_MAG_COEFFS", p_mag))

print(f"\nWrote to {OUT_DIR}:")
for fn in sorted(os.listdir(OUT_DIR)):
    print(f"  {fn:28s} {os.path.getsize(os.path.join(OUT_DIR, fn)):>9,d} bytes")

Stimulus vectors: 26707 total (137 in the corner-case subset)

Integer golden models match Phase 2 functions EXACTLY on all 26707 vectors (cordic, lut, poly).

=== Independent float64 reference (numpy arctan2/hypot on the same integer inputs) ===
 method |       bucket |      n | max |mag err| (LSB) | max mag err % | max ang err (deg)
--------------------------------------------------------------------------------------------
 cordic |   |v|<16 LSB |    792 |                8.00 |      800.0000 |           36.4401
 cordic |   16-256 LSB |   5518 |                7.96 |       42.6399 |            7.0136
 cordic |    >=256 LSB |  20396 |                8.07 |        2.5678 |            0.7449
    lut |   |v|<16 LSB |    792 |                1.00 |       29.2893 |            0.0236
    lut |   16-256 LSB |   5518 |                1.02 |        5.8824 |            0.0244
    lut |    >=256 LSB |  20396 |                6.55 |        0.3982 |            0.0258
   poly |   |v|<16 LSB |    79

In [ ]:
# =============================================================================
# PHASE 3 / STEP 1b: CORDIC guard-bit variant - golden vectors
#
# Run AFTER the Step 1 cell (reuses its stimulus set, ROMs and helpers).
#
# Idea: the baseline CORDIC shifts right every iteration and throws the shifted-out
# bits away, so tiny inputs lose relative precision. A guard-bit variant shifts the
# input LEFT by G bits first (extra fractional bits), runs the same loop, and
# shifts back at the end.
#
# IMPORTANT: this variant is NOT the Phase 2 model, so there is nothing to
# "match exactly". Its answer key is this integer model, and its correctness is
# judged against the independent float64 reference (numpy) below.
# =============================================================================
import numpy as np

G = 3    # guard bits

def cordic_int_g(xq, yq, G, track=None):
    x, y, off = xq << G, yq << G, 0
    if x < 0:
        x, y = -x, -y
        off = A180_C if y >= 0 else -A180_C
    acc = 0
    for i in range(N_ITER):
        xs_, ys_ = x >> i, y >> i
        if y < 0:
            x, y = x - ys_, y + xs_; acc -= atan_rom[i]
        else:
            x, y = x + ys_, y - xs_; acc += atan_rom[i]
        if track is not None:
            track['xy'] = max(track['xy'], abs(x), abs(y))
    mag_q = ((x * gain_recip) >> F) >> G
    a = acc + off
    if a >= A180_C:    a -= 2 * A180_C          # same wrap as the baseline
    elif a < -A180_C:  a += 2 * A180_C
    return mag_q, a

# sanity: with G=0 this must reproduce the baseline results exactly
b_mag, b_ang, _ = results['cordic']
for i, (x, y) in enumerate(stim):
    m0, a0 = cordic_int_g(x, y, 0)
    assert (m0, a0) == (b_mag[i], b_ang[i]), f"G=0 does not match baseline at {(x, y)}"
print("G=0 reproduces the validated baseline exactly (model is consistent).")

track = {'xy': 0}
g_mag = np.empty(len(stim), dtype=np.int64)
g_ang = np.empty(len(stim), dtype=np.int64)
for i, (x, y) in enumerate(stim):
    g_mag[i], g_ang[i] = cordic_int_g(x, y, G, track)

print(f"\n=== CORDIC baseline vs G={G} guard bits, against float64 reference ===")
print(f"{'variant':>9} | {'bucket':>12} | {'n':>6} | {'max |mag err| LSB':>17} | "
      f"{'max mag err %':>13} | {'max ang err deg':>15}")
print("-" * 88)
for label, mg, ag in (("baseline", b_mag, b_ang), (f"G={G}", g_mag, g_ang)):
    for b in range(3):
        sel = (mag_bucket == b) & ~is_zero
        merr = np.abs(mg[sel] - true_mag_lsb[sel])
        aerr = np.array([wrap_err(a / 2**AF, t) for a, t in zip(ag[sel], true_ang[sel])])
        print(f"{label:>9} | {bucket_names[b]:>12} | {sel.sum():>6} | {merr.max():>17.2f} | "
              f"{(merr / true_mag_lsb[sel]).max() * 100:>13.4f} | {aerr.max():>15.4f}")

print(f"\nG={G} x/y datapath: max |value| {track['xy']} -> "
      f"{signed_bits(-track['xy'], track['xy'])} signed bits "
      f"(baseline needs 18; HLS variant uses 18+G = {18 + G})")

for name, mg, ag in (("cordic_g%d" % G, g_mag, g_ang),):
    rows = list(zip(xs, ys, mg, ag, np.where(is_zero, 0, 1)))
    write_csv(f"{OUT_DIR}/{name}_full.csv", rows, "x_q,y_q,mag_q,ang_q,ang_check")
    write_csv(f"{OUT_DIR}/{name}_corner.csv", rows[:n_corner], "x_q,y_q,mag_q,ang_q,ang_check")
    print(f"Wrote {name}_full.csv ({len(rows)} vectors) and {name}_corner.csv ({n_corner}) to {OUT_DIR}")

G=0 reproduces the validated baseline exactly (model is consistent).

=== CORDIC baseline vs G=3 guard bits, against float64 reference ===
  variant |       bucket |      n | max |mag err| LSB | max mag err % | max ang err deg
----------------------------------------------------------------------------------------
 baseline |   |v|<16 LSB |    792 |              8.00 |      800.0000 |         36.4401
 baseline |   16-256 LSB |   5518 |              7.96 |       42.6399 |          7.0136
 baseline |    >=256 LSB |  20396 |              8.07 |        2.5678 |          0.7449
      G=3 |   |v|<16 LSB |    792 |              1.00 |      100.0000 |          9.8750
      G=3 |   16-256 LSB |   5518 |              1.07 |        5.7010 |          1.3335
      G=3 |    >=256 LSB |  20396 |              3.76 |        0.3699 |          0.0980

G=3 x/y datapath: max |value| 610500 -> 21 signed bits (baseline needs 18; HLS variant uses 18+G = 21)
Wrote cordic_g3_full.csv (26707 vectors) and cordic_

In [ ]:
# =============================================================================
# PHASE 3 / STEP 2: CIC + compensating-FIR golden vectors (decimation 100 AND 128)
#
# Needs these Phase 0/2 definitions already in this Colab session:
#   design_compensating_fir (and cic_frequency_response)   [Phase 0 cell]
#   quantize, cic_required_bit_growth, cic_decimate_fixed,
#   quantize_fir_taps, fir_apply_fixed                      [Phase 2 CIC/FIR cell]
# (Run this AFTER Steps 1/1b; it only needs the Phase 0/2 cells, not their outputs.)
#
# Why two decimation factors: 100 is what Phase 2 validated, but 100^4 is not a
# power of two, so the CIC rescale is a constant DIVIDE. 128^4 = 2^28, a free shift.
# Both are exported so Vivado can show the real cost of the difference.
#
# What it does, per decimation R in (100, 128):
#   1. Designs the 21-tap compensating FIR with YOUR design function and quantizes
#      it to Q2.14 exactly as in Phase 2.
#   2. Builds stimulus RUNS (each run starts with a reset): zeros, max DC (long
#      enough to wrap the integrators), alternating full scale, impulses, step,
#      sawtooth, full-scale sines, random, and run lengths around R.
#   3. Integer golden models (CIC, FIR, chain). Asserts each equals the Phase 2
#      functions EXACTLY, and equals an independent reference (CIC: direct
#      convolution with the equivalent boxcar^4 filter; FIR: scipy lfilter).
#   4. Re-runs the CIC with registers wrapped to the Hogenauer width to prove that
#      width is enough on every stimulus (and reports the true peak).
#   5. Writes CSVs (rst,x_q,valid,y_q) + a parameter header for the HLS code.
# =============================================================================
import os
import numpy as np
from scipy.signal import lfilter

N_ST, D_DL = 4, 1                 # CIC stages, comb delay
IN_W, IN_F = 16, 13               # Q3.13 samples
TAP_W, TAP_F = 16, 14             # taps Q2.14
NTAPS = 21
SEED = 2026
XMIN, XMAX = -(1 << (IN_W - 1)), (1 << (IN_W - 1)) - 1

OUT_DIR = ('/content/drive/MyDrive/lockin_project/golden_vectors'
           if os.path.ismount('/content/drive') else '/content/golden_vectors')
os.makedirs(OUT_DIR, exist_ok=True)

def wrap_signed(v, w):
    v &= (1 << w) - 1
    return v - (1 << w) if v >> (w - 1) else v

# ---------------- integer golden models (Python ints: exact, no overflow) -----
def cic_run_int(x, R, wrap_bits=None):
    """Streaming CIC for ONE run. Integrators see every sample; a sample is output
    when the phase counter is 0 (first sample included), exactly like y[::R].
    wrap_bits: emulate fixed-width two's-complement registers."""
    W = (lambda v: wrap_signed(v, wrap_bits)) if wrap_bits else (lambda v: v)
    I = [0] * N_ST
    Cd = [0] * N_ST
    out, cnt = [], 0                                   # (valid, y_q) per input sample
    gain = R ** N_ST
    comb_min = comb_max = 0
    for s in x:
        v = int(s)
        for k in range(N_ST):
            I[k] = W(I[k] + v); v = I[k]
        if cnt == 0:
            c = v
            for k in range(N_ST):
                d = W(c - Cd[k]); Cd[k] = c; c = d
            comb_min, comb_max = min(comb_min, c), max(comb_max, c)
            out.append((1, c // gain))                 # floor division (== >> for R=128)
        else:
            out.append((0, 0))
        cnt = 0 if cnt == R - 1 else cnt + 1
    return out, (comb_min, comb_max)

def fir_run_int(x, taps_q):
    hist = [0] * len(taps_q)
    out = []
    for s in x:
        hist = [int(s)] + hist[:-1]
        out.append(sum(t * h for t, h in zip(taps_q, hist)) >> TAP_F)
    return out

def chain_run_int(x, R, taps_q):
    cic, _ = cic_run_int(x, R)
    ys = [y for v, y in cic if v]
    f = fir_run_int(ys, taps_q)
    out, j = [], 0
    for v, _ in cic:
        if v: out.append((1, f[j])); j += 1
        else: out.append((0, 0))
    return out

# ---------------- stimulus ----------------------------------------------------
def sine_int(n, f_norm, phase=0.3, amp=XMAX):
    return np.clip(np.round(amp * np.sin(2 * np.pi * f_norm * np.arange(n) + phase)),
                   XMIN, XMAX).astype(np.int64)

def cic_runs(R):
    rng = np.random.default_rng(SEED)
    rn = lambda lo, hi, n: rng.integers(lo, hi + 1, n).astype(np.int64)
    runs = [("zeros", np.zeros(3 * R, np.int64)),
            ("dc_max", np.full(48 * R, XMAX, np.int64)),      # long: integrators WRAP
            ("dc_min", np.full(48 * R, XMIN, np.int64)),
            ("alt_fullscale", np.where(np.arange(20 * R) % 2 == 0, XMAX, XMIN).astype(np.int64)),
            ("impulse_pos", np.r_[XMAX, np.zeros(12 * R - 1)].astype(np.int64)),
            ("impulse_off37", np.r_[np.zeros(37), XMIN, np.zeros(12 * R - 38)].astype(np.int64)),
            ("step", np.r_[np.zeros(5 * R), np.full(7 * R, XMAX)].astype(np.int64)),
            ("sawtooth", ((np.arange(20 * R) * 517) % 65536 - 32768).astype(np.int64))]
    for tag, f in (("sine_low", 0.1 / (2 * R)), ("sine_pass_edge", 0.9 * 0.5 / R),
                   ("sine_out_nyq", 0.5 / R), ("sine_null", 1.0 / R),
                   ("sine_alias", 1.5 / R), ("sine_quarter", 0.25), ("sine_near_nyq", 0.4999)):
        runs.append((tag, sine_int(30 * R, f)))
    runs += [("rand_full", rn(XMIN, XMAX, 40 * R)), ("rand_small", rn(-16, 16, 20 * R)),
             ("rand_pm1", rn(-1, 1, 10 * R))]
    for L in (1, 2, R - 1, R, R + 1, 2 * R + 1):
        runs.append((f"edge_len_{L}", rn(XMIN, XMAX, L)))
    return runs

def fir_runs(taps_q):
    rng = np.random.default_rng(SEED + 1)
    rn = lambda lo, hi, n: rng.integers(lo, hi + 1, n).astype(np.int64)
    worst = lambda sgn: np.array([XMAX if (t * sgn) > 0 else XMIN
                                  for t in taps_q[::-1]], np.int64)   # aligns to the LAST output
    runs = [("zeros", np.zeros(30, np.int64)),
            ("impulse_pos", np.r_[XMAX, np.zeros(59)].astype(np.int64)),
            ("impulse_neg", np.r_[XMIN, np.zeros(59)].astype(np.int64)),
            ("step", np.r_[np.zeros(10), np.full(60, XMAX)].astype(np.int64)),
            ("dc_max", np.full(100, XMAX, np.int64)), ("dc_min", np.full(100, XMIN, np.int64)),
            ("alt_fullscale", np.where(np.arange(200) % 2 == 0, XMAX, XMIN).astype(np.int64)),
            ("worst_case_pos", worst(+1)), ("worst_case_neg", worst(-1))]
    for tag, f in (("sine_0.01", .01), ("sine_0.1", .1), ("sine_0.25", .25),
                   ("sine_0.4", .4), ("sine_0.49", .49)):
        runs.append((tag, sine_int(200, f)))
    runs += [("rand_full", rn(XMIN, XMAX, 5000)), ("rand_small", rn(-16, 16, 1000))]
    for L in (1, 2, NTAPS - 1, NTAPS, NTAPS + 1):
        runs.append((f"edge_len_{L}", rn(XMIN, XMAX, L)))
    return runs

def write_stream_csv(path, runs, outs):
    n = 0
    with open(path, 'w') as f:
        f.write("rst,x_q,valid,y_q\n")
        for (name, x), o in zip(runs, outs):
            for j, (s, (v, y)) in enumerate(zip(x, o)):
                f.write(f"{1 if j == 0 else 0},{int(s)},{v},{int(y) if v else 0}\n"); n += 1
    return n

# ---------------- main loop ---------------------------------------------------
hdr = ["// Auto-generated from the validated Phase 2 Python model. Do not hand-edit.",
       "#pragma once",
       f"#define CIC_N_STAGES {N_ST}", f"#define CIC_DIFF_DELAY {D_DL}",
       f"#define FIR_NTAPS {NTAPS}", f"#define FIR_TAP_FRAC {TAP_F}", f"#define FIR_TAP_WORD {TAP_W}",
       f"#define IN_FRAC {IN_F}"]
summary = []
for R in (100, 128):
    print(f"\n================ decimation R = {R} ================")
    taps_f = design_compensating_fir(R, N_ST, D_DL, NTAPS, 2000)
    taps_q = [int(t) for t in quantize_fir_taps(taps_f, TAP_F, TAP_W)]
    sym = all(taps_q[i] == taps_q[-1 - i] for i in range(NTAPS))
    print(f"FIR taps (Q2.{TAP_F}): {taps_q}   exactly symmetric: {sym}")
    acc_w = int(cic_required_bit_growth(IN_W, N_ST, R, D_DL))

    # ---- CIC -----
    runs = cic_runs(R)
    outs, peaks = [], []
    for name, x in runs:
        o, pk = cic_run_int(x, R)
        # (a) equals the Phase 2 function exactly
        p2 = cic_decimate_fixed(x / 2**IN_F, R, N_ST, D_DL, IN_F, IN_W)
        mine = np.array([y for v, y in o if v])
        assert np.array_equal(np.round(p2 * 2**IN_F).astype(np.int64), mine), f"CIC vs Phase 2: {name}"
        # (b) equals an independent reference: boxcar^4 convolution, decimate, floor
        h = np.ones(1, np.int64)
        for _ in range(N_ST): h = np.convolve(h, np.ones(R, np.int64))
        ref = (np.convolve(x, h)[:len(x)][::R]) // (R ** N_ST)
        assert np.array_equal(ref, mine), f"CIC vs boxcar^4 convolution: {name}"
        # (c) fixed-width registers (Hogenauer width) give the identical stream
        ow, _ = cic_run_int(x, R, wrap_bits=acc_w)
        assert ow == o, f"CIC wrapped to {acc_w} bits differs: {name}"
        assert mine.min() >= XMIN and mine.max() <= XMAX, f"CIC output exceeds 16 bits: {name}"
        outs.append(o); peaks.append(pk)
    n_in = write_stream_csv(f"{OUT_DIR}/cic_R{R}_stream.csv", runs, outs)
    # does a register 1 bit narrower still work? (is the bound tight or conservative?)
    ok_narrow = all(cic_run_int(x, R, wrap_bits=acc_w - 1)[0] == o for (n_, x), o in zip(runs, outs))
    lo_, hi_ = min(p[0] for p in peaks), max(p[1] for p in peaks)
    need = max(hi_.bit_length() + 1, (-lo_ - 1).bit_length() + 1 if lo_ < 0 else 1)
    print(f"CIC: {len(runs)} runs / {n_in} samples. == Phase 2, == boxcar^4 reference, and "
          f"identical with {acc_w}-bit wrapping registers.")
    print(f"     Hogenauer width {acc_w} bits; true comb range [{lo_}, {hi_}] needs "
          f"{need} signed bits; {acc_w - 1}-bit registers "
          f"{'STILL pass' if ok_narrow else 'FAIL'} on these stimuli.")

    # ---- FIR -----
    fruns = fir_runs(taps_q)
    fouts = []
    S = sum(abs(t) for t in taps_q)
    Sp = sum(t for t in taps_q if t > 0); Sn = sum(-t for t in taps_q if t < 0)
    hi_b = (XMAX * Sp + (-XMIN) * Sn) >> TAP_F          # most positive possible output
    lo_b = (XMIN * Sp - XMAX * Sn) >> TAP_F             # most negative possible output
    bound = hi_b
    out_w = max(hi_b.bit_length() + 1, (-lo_b - 1).bit_length() + 1)
    peak_f = 0
    for name, x in fruns:
        y = fir_run_int(x, taps_q)
        p2 = fir_apply_fixed(x / 2**IN_F, taps_q, TAP_F, TAP_W, IN_F, IN_W)
        assert np.array_equal(np.round(p2 * 2**IN_F).astype(np.int64), np.array(y)), f"FIR vs Phase 2: {name}"
        ref = np.floor(lfilter(np.array(taps_q, float), [1.0], x.astype(float)) / 2**TAP_F)
        assert np.array_equal(ref.astype(np.int64), np.array(y)), f"FIR vs scipy lfilter: {name}"
        peak_f = max(peak_f, max(abs(v) for v in y))
        assert lo_b <= min(y) and max(y) <= hi_b, f"FIR output outside analytic bound: {name}"
        fouts.append([(1, v) for v in y])
    n_f = write_stream_csv(f"{OUT_DIR}/fir_R{R}_stream.csv", fruns, fouts)
    print(f"FIR: {len(fruns)} runs / {n_f} samples. == Phase 2 and == scipy lfilter, bit-exact.")
    print(f"     sum|taps| = {S/2**TAP_F:.4f} -> worst-case output range [{lo_b}, {hi_b}] needs {out_w} signed bits; "
          f"observed peak {peak_f}. (output is NOT limited to 16 bits)")

    # ---- chain -----
    couts = [chain_run_int(x, R, taps_q) for _, x in runs]
    n_c = write_stream_csv(f"{OUT_DIR}/chain_R{R}_stream.csv", runs, couts)
    print(f"Chain (CIC -> FIR): {n_c} input samples written.")

    hdr += [f"#define CIC_ACC_W_R{R} {acc_w}", f"#define FIR_OUT_W_R{R} {out_w}",
            f"static const int FIR_TAPS_R{R}[{NTAPS}] = {{{','.join(map(str, taps_q))}}};"]
    summary.append((R, acc_w, out_w, sym))

with open(f"{OUT_DIR}/cic_fir_params.h", 'w') as f:
    f.write("\n".join(hdr) + "\n")
print(f"\nWrote to {OUT_DIR}:")
for fn in sorted(os.listdir(OUT_DIR)):
    if fn.startswith(('cic', 'fir', 'chain')):
        print(f"  {fn:28s} {os.path.getsize(os.path.join(OUT_DIR, fn)):>10,d} bytes")


================ decimation R = 100 ================
FIR taps (Q2.14): [141, -140, 59, 111, -339, 540, -527, -80, 2185, -8536, 29557, -8536, 2185, -80, -527, 540, -339, 111, 59, -140, 141]   exactly symmetric: True
CIC: 24 runs / 46004 samples. == Phase 2, == boxcar^4 reference, and identical with 43-bit wrapping registers.
     Hogenauer width 43 bits; true comb range [-3276800000000, 3276700000000] needs 43 signed bits; 42-bit registers FAIL on these stimuli.
FIR: 21 runs / 7728 samples. == Phase 2 and == scipy lfilter, bit-exact.
     sum|taps| = 3.3492 -> worst-case output range [-109745, 109743] needs 18 signed bits; observed peak 109745. (output is NOT limited to 16 bits)
Chain (CIC -> FIR): 46004 input samples written.

================ decimation R = 128 ================
FIR taps (Q2.14): [141, -140, 59, 111, -340, 540, -526, -80, 2185, -8534, 29555, -8534, 2185, -80, -526, 540, -340, 111, 59, -140, 141]   exactly symmetric: True
CIC: 24 runs / 58884 samples. == Phase 2, == bo

In [ ]:
# =============================================================================
# PHASE 3 / STEP 3: DDS (sine + cosine reference generator) - golden vectors
#
# Needs the Phase 0 cell already run in this Colab session (it defines
# DDSAccumulator). Nothing else is required.
#
# What a DDS is: a 32-bit "phase" counter that adds a fixed number (the phase
# increment) on every clock. The top 10 bits of the counter pick one entry of a
# 1024-entry sine table. The increment sets the output frequency:
#     f_out = increment * f_clk / 2^32
# Cosine is the same table read a quarter turn (90 degrees) ahead, so ONE table
# gives both references the lock-in mixer needs.
#
# This cell:
#   1. Builds an integer model of the DDS and asserts it equals YOUR Phase 0
#      DDSAccumulator.generate() exactly (sine, and cosine via the +90 deg start
#      phase used in your Step 3 cross-check) on every test run.
#   2. Checks every one of the 1024 table entries gets exercised by the tests.
#   3. Measures the spectral purity (spur level) of the 1024-entry table, so the
#      limitation is known and can be reported.
#   4. Writes dds_stream.csv (rst, phase_inc, sin_q, cos_q) and dds_params.h.
# =============================================================================
import os
import numpy as np

FCLK = 200_000                       # only used for the frequency-in-Hz printout
PHASE_BITS, LUT_BITS = 32, 10
SHIFT = PHASE_BITS - LUT_BITS        # 22: top 10 bits index the table
QUARTER = 1 << (PHASE_BITS - 2)      # +90 degrees = 2^30
MASK = (1 << PHASE_BITS) - 1
SEED = 7

OUT_DIR = ('/content/drive/MyDrive/lockin_project/golden_vectors'
           if os.path.ismount('/content/drive') else '/content/golden_vectors')
os.makedirs(OUT_DIR, exist_ok=True)

ref = DDSAccumulator(FCLK, PHASE_BITS, LUT_BITS)          # YOUR validated model
ROM = [int(v) for v in ref.sine_lut]

# ---------------- integer model (what the hardware does each clock) ----------
def dds_run_int(incs):
    """incs: one phase increment per clock. A run starts at phase 0 (reset)."""
    phase, out = 0, []
    for inc in incs:
        s = ROM[(phase >> SHIFT) & (len(ROM) - 1)]
        c = ROM[(((phase + QUARTER) & MASK) >> SHIFT) & (len(ROM) - 1)]
        out.append((s, c))
        phase = (phase + int(inc)) & MASK
    return out

# ---------------- stimulus runs: (name, list of per-clock increments) ---------
rng = np.random.default_rng(SEED)
inc_1khz = ref.freq_to_increment(1_000)                      # your Step 3 case
runs = [("dc_inc0", [0] * 16),
        ("1kHz_notebook_case", [inc_1khz] * 2000),
        ("one_table_step_per_clock", [1 << SHIFT] * 2100),   # visits every entry
        ("nyquist_2^31", [1 << 31] * 64),
        ("max_inc_2^32-1", [MASK] * 2100),                   # steps backwards by 1
        ("smallest_inc_1", [1] * 64),
        ("straddle_minus1", [(1 << SHIFT) - 1] * 4200),      # lands on table edges
        ("straddle_plus1", [(1 << SHIFT) + 1] * 4200),
        ("quarter_turn_2^30", [QUARTER] * 64),
        ("single_clock_run", [inc_1khz])]
for k in range(10):
    runs.append((f"random_inc_{k}", [int(rng.integers(0, 1 << PHASE_BITS))] * 3000))
hop = []
for inc in (inc_1khz, 5 * inc_1khz, 123456789, 0, 3999999999, 1 << SHIFT, 777):
    hop += [inc] * 150                                       # frequency changes mid-run
runs.append(("frequency_hops_phase_continuous", hop))

# ---------------- checks against YOUR Phase 0 class ---------------------------
def run_via_class(incs, start_phase):
    d = DDSAccumulator(FCLK, PHASE_BITS, LUT_BITS)
    d.phase_reg = start_phase
    out, i = [], 0
    while i < len(incs):                                     # group equal increments
        j = i
        while j < len(incs) and incs[j] == incs[i]: j += 1
        f = incs[i] * FCLK / 2 ** PHASE_BITS                 # class rounds this back to incs[i]
        assert d.freq_to_increment(f) == incs[i], "class would not reproduce this increment"
        samples, _ = d.generate(f, j - i)
        out += [int(v) for v in samples]
        i = j
    return out

seen_s, seen_c, n_total = set(), set(), 0
outs = []
for name, incs in runs:
    o = dds_run_int(incs)
    sin_cls = run_via_class(incs, 0)                         # sine: start phase 0
    cos_cls = run_via_class(incs, QUARTER)                   # cosine: start +90 deg (your Step 3)
    assert [a for a, _ in o] == sin_cls, f"sine differs from Phase 0 DDSAccumulator: {name}"
    assert [b for _, b in o] == cos_cls, f"cosine differs from Phase 0 DDSAccumulator: {name}"
    assert all(-32768 <= v <= 32767 for p in o for v in p)
    ph = 0
    for inc in incs:
        seen_s.add((ph >> SHIFT) & 1023); seen_c.add((((ph + QUARTER) & MASK) >> SHIFT) & 1023)
        ph = (ph + inc) & MASK
    outs.append(o); n_total += len(incs)
print(f"{len(runs)} runs / {n_total} clocks. Integer model == Phase 0 DDSAccumulator, "
      f"exactly, for sine AND cosine on every run.")
print(f"Table entries exercised: sine {len(seen_s)}/1024, cosine {len(seen_c)}/1024")
assert len(seen_s) == 1024 and len(seen_c) == 1024

# ---------------- spectral purity of the table (informational) ----------------
N = 65536
cycles = 1311                                                # integer cycles -> no leakage
inc = cycles * (1 << PHASE_BITS) // N
sig = np.array([a for a, _ in dds_run_int([inc] * N)], dtype=float)
spec = np.abs(np.fft.rfft(sig * np.blackman(N)))
k0 = int(np.argmax(spec[1:])) + 1
carrier = spec[k0 - 4:k0 + 5].max()
spec2 = spec.copy(); spec2[max(0, k0 - 8):k0 + 9] = 0; spec2[:3] = 0
sfdr = 20 * np.log10(carrier / spec2.max())
print(f"\nSpur level of the 1024-entry table (no interpolation): SFDR = {sfdr:.1f} dBc "
      f"(largest unwanted tone vs the wanted sine)")
print("  -> a lock-in's narrow filter rejects spurs away from the reference frequency, "
      "but this sets a floor for strong off-frequency interferers.")

# ---------------- write files -------------------------------------------------
with open(f"{OUT_DIR}/dds_stream.csv", 'w') as f:
    f.write("rst,phase_inc,sin_q,cos_q\n")
    for (name, incs), o in zip(runs, outs):
        for j, (inc_, (s, c)) in enumerate(zip(incs, o)):
            f.write(f"{1 if j == 0 else 0},{int(inc_)},{s},{c}\n")
with open(f"{OUT_DIR}/dds_params.h", 'w') as f:
    f.write("// Auto-generated from the validated Phase 0 DDSAccumulator. Do not hand-edit.\n#pragma once\n")
    f.write(f"#define DDS_PHASE_BITS {PHASE_BITS}\n#define DDS_LUT_BITS {LUT_BITS}\n")
    f.write(f"static const int DDS_SINE_ROM[{len(ROM)}] = {{{','.join(map(str, ROM))}}};\n")
print(f"\nWrote dds_stream.csv ({n_total} rows) and dds_params.h to {OUT_DIR}")

21 runs / 45859 clocks. Integer model == Phase 0 DDSAccumulator, exactly, for sine AND cosine on every run.
Table entries exercised: sine 1024/1024, cosine 1024/1024

Spur level of the 1024-entry table (no interpolation): SFDR = 60.2 dBc (largest unwanted tone vs the wanted sine)
  -> a lock-in's narrow filter rejects spurs away from the reference frequency, but this sets a floor for strong off-frequency interferers.

Wrote dds_stream.csv (45859 rows) and dds_params.h to /content/drive/MyDrive/lockin_project/golden_vectors
